
# BR001 | Analysis Readiness Validation

**Project:** Nova Corporation — Executive Performance Review  
**Document Type:** Technical Validation Notebook  
**Status:** Draft — Pending Validation  
**Primary Reference:** IRDS-BR001  
**Supporting References:** Approved Business Clarifications CL001–CL007  
**Source:** Controlled Analytical Dataset

## Objective

Independently validate analytical populations, KPI calculations, approved clarification treatments, and data integrity before starting business analysis BQ01–BQ07.

## Validation Sections

1. Notebook Setup
2. Dataset Loading & Structural Validation
3. Analytical Population Validation
4. KPI Reconciliation
5. Clarification Compliance
6. Dimensional Integrity & Aggregation
7. KPI Readiness & Restrictions
8. Validation Summary & Execution Gate

## Execution Principles

- Preserve source and controlled analytical data.
- Independently reconstruct critical eligibility rules.
- Compare observed results against approved baselines.
- Record PASS, FAIL, or REVIEW for each validation control.
- Document discrepancies before applying any corrective action.
- Do not begin business interpretation until the execution gate is satisfied.



# 01 — Notebook Setup

**Objective:** Initialize the analytical validation environment, load the required libraries, and establish reproducible access to the controlled BR001 analytical dataset.

**Governing Reference:** IRDS-BR001 — Integrated Requirements & Data Specification.

## Validation Scope

- Import the required analytical libraries.
- Resolve the project directory and controlled dataset path.
- Load the approved cleaned analytical dataset.
- Confirm the execution environment and library availability.
- Establish a consistent starting point for subsequent validation controls.

## Acceptance Principles

- Use the controlled BR001 analytical dataset.
- Preserve the original dataset without modification.
- Avoid machine-specific absolute paths where practical.
- Ensure that subsequent validation steps can be reproduced.
- Report initialization or loading errors before proceeding.


In [1]:

# 01 — Notebook Setup
# Required libraries

from pathlib import Path

import pandas as pd
import numpy as np

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Notebook setup completed successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Notebook setup completed successfully.
Pandas version: 3.0.6
NumPy version: 2.5.3


In [2]:

# 01 — Notebook Setup
# Project paths

PROJECT_ROOT = Path.cwd().resolve().parents[2]

DATA_PATH = (
    PROJECT_ROOT
    / "04_data"
    / "clean"
    / "br001_clean_analytical_dataset.csv"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Dataset exists: {DATA_PATH.is_file()}")


Project root: C:\Users\Usuario\OneDrive\Escritorio\Nova_Corporation
Dataset path: C:\Users\Usuario\OneDrive\Escritorio\Nova_Corporation\04_data\clean\br001_clean_analytical_dataset.csv
Dataset exists: True



# 02 — Dataset Loading & Structural Validation

**Objective:** Verify that the controlled analytical dataset is structurally complete, readable, and suitable for subsequent population and financial validation.

**Governing Reference:** IRDS-BR001 — Integrated Requirements & Data Specification; DP-2026-001 — Approved Data Release.

## Validation Scope

- Confirm dataset availability and successful loading.
- Verify the expected record and column counts.
- Validate the presence of required analytical fields.
- Inspect column names, data types, and null values.
- Confirm analytical row key completeness and uniqueness.
- Verify reporting date coverage for H1 2025 and H1 2026.
- Review transaction status distribution.

## Acceptance Principles

- Preserve the expected 23,530 records and 47 columns.
- Maintain all required fields for downstream analysis.
- Ensure that analytical row keys are unique and non-null.
- Restrict the reporting scope to the approved comparison periods.
- Identify structural discrepancies before calculating KPIs.
- Do not interpret structural completeness as proof that all source-data defects are resolved.


In [3]:

# 02 — Dataset Loading & Structural Validation
# Load the controlled analytical dataset

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Duplicate column names: {df.columns.duplicated().sum()}")

display(df.head(5))


Dataset loaded successfully.
Rows: 23,530
Columns: 47
Duplicate column names: 0


,source_rowid,sales_line_id,order_id,order_date,customer_id,product_id,channel_id,region_id,quantity,unit_price,unit_cost,discount_pct,gross_revenue,discount_amount,net_revenue,total_cost,gross_profit,payment_method,order_status,sales_line_id_source,analytical_row_key,line_id_collision_flag,missing_customer_flag,missing_channel_flag,customer_attribution,channel_attribution,payment_method_source,unmapped_payment_flag,payment_method_reporting,invalid_quantity_flag,invalid_unit_price_flag,invalid_unit_cost_flag,invalid_discount_pct_flag,invalid_numeric_flag,numeric_quality_eligible,order_status_source,certified_sales_eligible,provisional_return_inclusive_eligible,high_total_cost_outlier_flag,negative_gross_profit_flag,high_cost_negative_profit_flag,order_id_source,order_customer_conflict_flag,order_date_conflict_flag,order_identity_conflict_flag,order_key_verified,order_kpi_eligible
0,1,SL0000001,O0000001,2025-02-11,C00874,P0142,CH01,R004,2,493.66,232.67,0.20,987.32,197.46,789.86,465.34,324.52,Cash,Completed,SL0000001,180c6f846d151ea0-000001,False,False,False,C00874,CH01,Cash,False,Cash,False,False,False,False,False,True,Completed,True,True,False,False,False,O0000001,False,False,False,False,False
1,2,SL0000002,O0000002,2025-04-27,C01154,P0138,CH03,R004,3,266.22,174.04,0.15,798.66,119.80,678.86,522.12,156.74,Bank Transfer,Completed,SL0000002,180c6f846d151ea0-000002,False,False,False,C01154,CH03,Bank Transfer,False,Bank Transfer,False,False,False,False,False,True,Completed,True,True,False,False,False,O0000002,True,True,True,False,False
2,3,SL0000003,O0000002,2025-03-24,C01565,P0127,CH04,R003,1,62.08,41.98,0.35,62.08,21.73,40.35,41.98,-1.63,Digital Wallet,Completed,SL0000003,180c6f846d151ea0-000003,False,False,False,C01565,CH04,Digital Wallet,False,Digital Wallet,False,False,False,False,False,True,Completed,True,True,False,True,False,O0000002,True,True,True,False,False
3,4,SL0000004,O0000002,2025-04-24,C02110,P0129,CH04,R006,1,205.30,123.86,0.00,205.30,0.00,205.30,123.86,81.44,Cash,Completed,SL0000004,180c6f846d151ea0-000004,False,False,False,C02110,CH04,Cash,False,Cash,False,False,False,False,False,True,Completed,True,True,False,False,False,O0000002,True,True,True,False,False
4,5,SL0000005,O0000003,2025-02-07,C00466,P0158,CH02,R001,2,209.99,119.03,0.10,419.98,42.00,377.98,238.06,139.92,Cash,Completed,SL0000005,180c6f846d151ea0-000005,False,False,False,C00466,CH02,Cash,False,Cash,False,False,False,False,False,True,Completed,True,True,False,False,False,O0000003,True,True,True,False,False


In [4]:

# 02 — Dataset Loading & Structural Validation
# Schema and data type inspection

required_columns = [
    "source_rowid",
    "sales_line_id",
    "order_id",
    "order_date",
    "customer_id",
    "product_id",
    "channel_id",
    "region_id",
    "quantity",
    "unit_price",
    "unit_cost",
    "discount_pct",
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "total_cost",
    "gross_profit",
    "order_status",
    "analytical_row_key",
    "invalid_numeric_flag",
    "certified_sales_eligible",
    "provisional_return_inclusive_eligible",
    "order_kpi_eligible"
]

missing_columns = sorted(set(required_columns) - set(df.columns))

schema_result = pd.DataFrame({
    "column": required_columns,
    "present": [col in df.columns for col in required_columns],
    "dtype": [
        str(df[col].dtype) if col in df.columns else "MISSING"
        for col in required_columns
    ]
})

display(schema_result)

print(f"Required columns: {len(required_columns)}")
print(f"Missing columns: {len(missing_columns)}")
print(f"Missing column names: {missing_columns}")


,column,present,dtype
0,source_rowid,True,int64
1,sales_line_id,True,str
2,order_id,True,str
3,order_date,True,str
4,customer_id,True,str
5,product_id,True,str
6,channel_id,True,str
7,region_id,True,str
8,quantity,True,int64
9,unit_price,True,float64


Required columns: 23
Missing columns: 0
Missing column names: []


In [5]:

# 02 — Dataset Loading & Structural Validation
# Data types and null inspection

numeric_columns = [
    "quantity", "unit_price", "unit_cost", "discount_pct",
    "gross_revenue", "discount_amount", "net_revenue",
    "total_cost", "gross_profit"
]

boolean_columns = [
    "invalid_numeric_flag",
    "certified_sales_eligible",
    "provisional_return_inclusive_eligible",
    "order_kpi_eligible"
]

validation_results = []

for col in numeric_columns:
    validation_results.append({
        "column": col,
        "expected_type": "numeric",
        "actual_type": str(df[col].dtype),
        "type_valid": pd.api.types.is_numeric_dtype(df[col]),
        "null_count": int(df[col].isna().sum())
    })

for col in boolean_columns:
    validation_results.append({
        "column": col,
        "expected_type": "boolean",
        "actual_type": str(df[col].dtype),
        "type_valid": pd.api.types.is_bool_dtype(df[col]),
        "null_count": int(df[col].isna().sum())
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("Invalid data types:", (~validation_df["type_valid"]).sum())
print("Total null values in checked columns:", validation_df["null_count"].sum())

print(
    "Null analytical row keys:",
    df["analytical_row_key"].isna().sum()
)

print(
    "Duplicate analytical row keys:",
    df["analytical_row_key"].duplicated().sum()
)


,column,expected_type,actual_type,type_valid,null_count
0,quantity,numeric,int64,True,0
1,unit_price,numeric,float64,True,0
2,unit_cost,numeric,float64,True,0
3,discount_pct,numeric,float64,True,0
4,gross_revenue,numeric,float64,True,0
5,discount_amount,numeric,float64,True,0
6,net_revenue,numeric,float64,True,0
7,total_cost,numeric,float64,True,0
8,gross_profit,numeric,float64,True,0
9,invalid_numeric_flag,boolean,bool,True,0


Invalid data types: 0
Total null values in checked columns: 0
Null analytical row keys: 0
Duplicate analytical row keys: 0


In [6]:

# 02 — Dataset Loading & Structural Validation
# Reporting period validation

parsed_dates = pd.to_datetime(df["order_date"], errors="coerce")

valid_period = (
    parsed_dates.dt.year.isin([2025, 2026])
    & parsed_dates.dt.month.between(1, 6)
)

date_checks = {
    "Unparseable dates": int(parsed_dates.isna().sum()),
    "Dates outside approved H1 periods": int(
        (parsed_dates.notna() & ~valid_period).sum()
    ),
    "Missing order dates": int(df["order_date"].isna().sum())
}

display(pd.DataFrame(
    date_checks.items(),
    columns=["Check", "Observed"]
))

print("\nRecords by reporting year:")
display(
    parsed_dates.dt.year.value_counts(dropna=False)
    .sort_index()
    .rename_axis("year")
    .reset_index(name="records")
)


,Check,Observed
0,Unparseable dates,0
1,Dates outside approved H1 periods,0
2,Missing order dates,0



Records by reporting year:


,year,records
0,2025,11250
1,2026,12280



# 03 — Analytical Population Validation

**Objective:** Independently reconstruct and validate the approved analytical populations, ensuring that transaction status and numeric validity rules are applied consistently.

**Governing Reference:** IRDS-BR001 — Integrated Requirements & Data Specification; BR001 — Business Clarification Response & Approved Cleaning Decisions.

## Validation Scope

- Reconstruct numeric validity rules.
- Validate the `invalid_numeric_flag` classification.
- Reconstruct certified sales eligibility.
- Reconstruct provisional return-inclusive eligibility.
- Verify eligibility flags against independently derived conditions.
- Reconcile included and excluded transaction populations.
- Confirm that unresolved order identity restrictions remain enforced.

## Acceptance Principles

- Include only valid `Completed` records in certified financial KPIs.
- Include valid `Completed` and `Partially Returned` records in provisional reporting.
- Exclude `Pending`, `Cancelled`, and numerically invalid records from both financial populations.
- Preserve excluded records in the controlled dataset.
- Maintain the distinction between certified and provisional populations.
- Do not certify restricted order-level KPIs without verified order identity.


In [7]:

# 03 — Analytical Population Validation
# Transaction status distribution

status_distribution = (
    df.groupby("order_status", dropna=False)
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

status_distribution["percentage"] = (
    status_distribution["record_count"]
    / len(df)
    * 100
).round(2)

display(status_distribution)

# Validate approved status domain
approved_statuses = {
    "Completed",
    "Partially Returned",
    "Pending",
    "Cancelled"
}

observed_statuses = set(df["order_status"].dropna().unique())

unexpected_statuses = observed_statuses - approved_statuses
missing_status_count = int(df["order_status"].isna().sum())

print(f"Total records: {len(df):,}")
print(f"Unexpected statuses: {sorted(unexpected_statuses)}")
print(f"Missing statuses: {missing_status_count}")
print(
    "Distribution reconciles:",
    status_distribution["record_count"].sum() == len(df)
)


,order_status,record_count,percentage
1,Completed,21345,90.71
2,Partially Returned,1075,4.57
0,Cancelled,767,3.26
3,Pending,343,1.46


Total records: 23,530
Unexpected statuses: []
Missing statuses: 0
Distribution reconciles: True


In [8]:

# 03 — Analytical Population Validation
# Independent numeric validity reconstruction — CL-005

numeric_fields = [
    "quantity", "unit_price", "unit_cost", "discount_pct"
]

numeric_values = df[numeric_fields].apply(
    pd.to_numeric, errors="coerce"
)

expected_invalid_numeric = (
    numeric_values.isna().any(axis=1)
    | (numeric_values["quantity"] <= 0)
    | (numeric_values["unit_price"] <= 0)
    | (numeric_values["unit_cost"] < 0)
    | ~numeric_values["discount_pct"].between(0, 1)
)

numeric_mismatches = (
    expected_invalid_numeric != df["invalid_numeric_flag"]
)

print(f"Expected invalid records: {expected_invalid_numeric.sum():,}")
print(f"Flagged invalid records: {df['invalid_numeric_flag'].sum():,}")
print(f"Flag mismatches: {numeric_mismatches.sum():,}")

display(
    df.loc[
        expected_invalid_numeric | numeric_mismatches,
        [
            "sales_line_id",
            "order_status",
            *numeric_fields,
            "invalid_numeric_flag"
        ]
    ]
)


Expected invalid records: 3
Flagged invalid records: 3
Flag mismatches: 0


,sales_line_id,order_status,quantity,unit_price,unit_cost,discount_pct,invalid_numeric_flag
12000,SL0012001,Completed,-2,502.70,330.31,0.00,True
15100,SL0015101,Completed,1,626.96,-4.75,0.10,True
18300,SL0018301,Completed,2,319.04,199.58,1.15,True


In [9]:

# 03 — Analytical Population Validation
# Independent eligibility reconstruction — CL-002 / CL-005

expected_certified = (
    df["order_status"].eq("Completed")
    & ~expected_invalid_numeric
)

expected_provisional = (
    df["order_status"].isin(["Completed", "Partially Returned"])
    & ~expected_invalid_numeric
)

population_results = pd.DataFrame([
    {
        "population": "Certified",
        "expected_count": int(expected_certified.sum()),
        "actual_count": int(df["certified_sales_eligible"].sum()),
        "mismatches": int(
            (expected_certified != df["certified_sales_eligible"]).sum()
        )
    },
    {
        "population": "Provisional",
        "expected_count": int(expected_provisional.sum()),
        "actual_count": int(
            df["provisional_return_inclusive_eligible"].sum()
        ),
        "mismatches": int(
            (
                expected_provisional
                != df["provisional_return_inclusive_eligible"]
            ).sum()
        )
    }
])

display(population_results)

print(
    "Numeric flag mismatches:",
    int(numeric_mismatches.sum())
)


,population,expected_count,actual_count,mismatches
0,Certified,21342,21342,0
1,Provisional,22417,22417,0


Numeric flag mismatches: 0


In [10]:

# 03 — Analytical Population Validation
# Population reconciliation bridge

population_bridge = pd.DataFrame([
    {
        "category": "Certified Completed",
        "records": int(expected_certified.sum())
    },
    {
        "category": "Valid Partially Returned",
        "records": int(
            (
                df["order_status"].eq("Partially Returned")
                & ~expected_invalid_numeric
            ).sum()
        )
    },
    {
        "category": "Invalid Numeric",
        "records": int(expected_invalid_numeric.sum())
    },
    {
        "category": "Pending",
        "records": int(df["order_status"].eq("Pending").sum())
    },
    {
        "category": "Cancelled",
        "records": int(df["order_status"].eq("Cancelled").sum())
    }
])

display(population_bridge)

bridge_total = int(population_bridge["records"].sum())

print(f"Bridge total: {bridge_total:,}")
print(f"Source records: {len(df):,}")
print(f"Reconciliation difference: {bridge_total - len(df):,}")


,category,records
0,Certified Completed,21342
1,Valid Partially Returned,1075
2,Invalid Numeric,3
3,Pending,343
4,Cancelled,767


Bridge total: 23,530
Source records: 23,530
Reconciliation difference: 0



# 04 — KPI Reconciliation

**Objective:** Verify that certified financial KPIs reconcile to approved control totals and remain internally consistent across transaction, annual, and monthly reporting levels.

**Governing Reference:** IRDS-BR001 — Integrated Requirements & Data Specification; BR001 — Approved Analytical Plan.

## Validation Scope

- Reconcile certified Gross Revenue.
- Reconcile certified Discount Amount.
- Reconcile certified Net Revenue.
- Reconcile certified Total Cost.
- Reconcile certified Gross Profit.
- Validate row-level financial arithmetic.
- Reconcile annual totals for H1 2025 and H1 2026.
- Reconcile monthly financial aggregations to certified totals.
- Confirm coverage of the 12 approved reporting months.

## Acceptance Principles

- Calculate certified KPIs exclusively from eligible records.
- Match approved financial control totals within the defined tolerance.
- Preserve financial arithmetic consistency.
- Ensure that annual and monthly aggregations reconcile to the certified population.
- Avoid combining overlapping certified and provisional populations.
- Keep Order Count, AOV, and other restricted order-level KPIs blocked until order identity is verified.


In [11]:

# 04 — KPI Reconciliation
# Certified financial KPI validation

certified_df = df.loc[expected_certified].copy()

financial_columns = [
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "total_cost",
    "gross_profit"
]

# Approved reconciliation baseline
control_totals = {
    "gross_revenue": 16068562.07,
    "discount_amount": 2504281.22,
    "net_revenue": 13564280.85,
    "total_cost": 9483638.54,
    "gross_profit": 4080642.31
}

# Independently calculate observed totals
observed_totals = certified_df[financial_columns].sum()

reconciliation = pd.DataFrame({
    "kpi": financial_columns,
    "expected": [control_totals[col] for col in financial_columns],
    "observed": [observed_totals[col] for col in financial_columns]
})

# Compare at reporting precision (2 decimal places)
reconciliation["difference"] = (
    reconciliation["observed"] - reconciliation["expected"]
).round(2)

reconciliation["status"] = np.where(
    reconciliation["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

display(reconciliation)

print(f"Certified records: {len(certified_df):,}")
print(f"Passed KPIs: {(reconciliation['status'] == 'PASS').sum()}")
print(f"Failed KPIs: {(reconciliation['status'] == 'FAIL').sum()}")


,kpi,expected,observed,difference,status
0,gross_revenue,"16,068,562.07","16,068,562.07",0.00,PASS
1,discount_amount,"2,504,281.22","2,504,281.22",-0.00,PASS
2,net_revenue,"13,564,280.85","13,564,280.85",0.00,PASS
3,total_cost,"9,483,638.54","9,483,638.54",0.00,PASS
4,gross_profit,"4,080,642.31","4,080,642.31",0.00,PASS


Certified records: 21,342
Passed KPIs: 5
Failed KPIs: 0


In [12]:

# 04 — KPI Reconciliation
# Row-level financial integrity validation

TOLERANCE = 0.01

revenue_difference = (
    certified_df["gross_revenue"]
    - certified_df["discount_amount"]
    - certified_df["net_revenue"]
)

profit_difference = (
    certified_df["net_revenue"]
    - certified_df["total_cost"]
    - certified_df["gross_profit"]
)

financial_integrity = pd.DataFrame([
    {
        "check": "Gross Revenue - Discount = Net Revenue",
        "records_checked": len(certified_df),
        "failed_records": int(
            (revenue_difference.abs() > TOLERANCE).sum()
        ),
        "max_difference": revenue_difference.abs().max()
    },
    {
        "check": "Net Revenue - Total Cost = Gross Profit",
        "records_checked": len(certified_df),
        "failed_records": int(
            (profit_difference.abs() > TOLERANCE).sum()
        ),
        "max_difference": profit_difference.abs().max()
    }
])

financial_integrity["status"] = np.where(
    financial_integrity["failed_records"] == 0,
    "PASS",
    "FAIL"
)

display(financial_integrity)


,check,records_checked,failed_records,max_difference,status
0,Gross Revenue - Discount = Net Revenue,21342,0,0.00,PASS
1,Net Revenue - Total Cost = Gross Profit,21342,0,0.00,PASS


In [13]:

# 04 — KPI Reconciliation
# Year-level financial reconciliation

certified_df["reporting_year"] = pd.to_datetime(
    certified_df["order_date"]
).dt.year

yearly_financials = (
    certified_df
    .groupby("reporting_year")[financial_columns]
    .sum()
    .round(2)
)

display(yearly_financials)

yearly_reconciliation = pd.DataFrame({
    "kpi": financial_columns,
    "yearly_sum": yearly_financials[financial_columns].sum().values,
    "certified_total": observed_totals[financial_columns].values
})

yearly_reconciliation["difference"] = (
    yearly_reconciliation["yearly_sum"]
    - yearly_reconciliation["certified_total"]
).round(2)

yearly_reconciliation["status"] = np.where(
    yearly_reconciliation["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

display(yearly_reconciliation)

print(
    "Year-level reconciliation:",
    "PASS" if yearly_reconciliation["status"].eq("PASS").all()
    else "FAIL"
)


,gross_revenue,discount_amount,net_revenue,total_cost,gross_profit
reporting_year,,,,,
2025,"7,764,554.70","1,089,539.31","6,675,015.39","4,459,069.63","2,215,945.76"
2026,"8,304,007.37","1,414,741.91","6,889,265.46","5,024,568.91","1,864,696.55"


,kpi,yearly_sum,certified_total,difference,status
0,gross_revenue,"16,068,562.07","16,068,562.07",0.00,PASS
1,discount_amount,"2,504,281.22","2,504,281.22",0.00,PASS
2,net_revenue,"13,564,280.85","13,564,280.85",0.00,PASS
3,total_cost,"9,483,638.54","9,483,638.54",0.00,PASS
4,gross_profit,"4,080,642.31","4,080,642.31",-0.00,PASS


Year-level reconciliation: PASS


In [14]:

# 04 — KPI Reconciliation
# Monthly financial aggregation validation

certified_df["reporting_month"] = pd.to_datetime(
    certified_df["order_date"]
).dt.month

monthly_financials = (
    certified_df
    .groupby(["reporting_year", "reporting_month"])[financial_columns]
    .sum()
    .round(2)
)

monthly_reconciliation = pd.DataFrame({
    "kpi": financial_columns,
    "monthly_sum": monthly_financials.sum().values,
    "certified_total": observed_totals[financial_columns].values
})

monthly_reconciliation["difference"] = (
    monthly_reconciliation["monthly_sum"]
    - monthly_reconciliation["certified_total"]
).round(2)

monthly_reconciliation["status"] = np.where(
    monthly_reconciliation["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

print(f"Reporting periods: {len(monthly_financials)}")
display(monthly_financials)
display(monthly_reconciliation)


Reporting periods: 12


gross_revenue  discount_amount  net_revenue  \
reporting_year reporting_month                                                
2025           1                 1,351,815.92       186,408.56 1,165,407.36   
               2                 1,181,083.82       162,548.60 1,018,535.22   
               3                 1,318,849.78       179,736.00 1,139,113.78   
               4                 1,295,176.28       182,039.00 1,113,137.28   
               5                 1,261,068.23       186,148.92 1,074,919.31   
               6                 1,356,560.67       192,658.23 1,163,902.44   
2026           1                 1,383,617.74       230,377.00 1,153,240.74   
               2                 1,366,557.76       242,179.31 1,124,378.45   
               3                 1,339,262.39       231,272.12 1,107,990.27   
               4                 1,421,939.95       239,482.20 1,182,457.75   
               5                 1,441,461.90       246,521.65 1,194,940.25   
               6                 1,351,167.63       224,909.63 1,126,258.00   

                                total_cost  gross_profit  
reporting_year reporting_month                            
2025           1                773,418.58    391,988.78  
               2                686,821.61    331,713.61  
               3                754,270.88    384,842.90  
               4                739,357.44    373,779.84  
               5                724,378.41    350,540.90  
               6                780,822.71    383,079.73  
2026           1                838,420.68    314,820.06  
               2                821,652.86    302,725.59  
               3                815,240.29    292,749.98  
               4                861,117.13    321,340.62  
               5                875,758.31    319,181.94  
               6                812,379.64    313,878.36

,kpi,monthly_sum,certified_total,difference,status
0,gross_revenue,"16,068,562.07","16,068,562.07",0.00,PASS
1,discount_amount,"2,504,281.22","2,504,281.22",0.00,PASS
2,net_revenue,"13,564,280.85","13,564,280.85",0.00,PASS
3,total_cost,"9,483,638.54","9,483,638.54",0.00,PASS
4,gross_profit,"4,080,642.31","4,080,642.31",0.00,PASS



# 05 — Clarification Compliance

**Objective:** Verify that the controlled analytical dataset implements the approved business clarification decisions without altering source facts or incorrectly certifying unresolved source defects.

**Governing Reference:** BR001 — Business Clarification Response & Approved Cleaning Decisions.

## Validation Scope

- CL-001: Order identity and restricted order KPIs.
- CL-002: Transaction-status eligibility and partial returns.
- CL-003: Payment method classification.
- CL-004: Duplicate sales-line identifiers.
- CL-005: Invalid numeric records.
- CL-006: Missing dimension attribution.
- CL-007: High-cost outliers and negative gross profit.

## Acceptance Principles

- Preserve original source values.
- Verify approved analytical treatments independently.
- Confirm record-level traceability.
- Distinguish reporting controls from source corrections.
- Keep unresolved source defects documented.
- Record PASS, FAIL, or REVIEW for each clarification.
- Do not treat validated analytical workarounds as resolution of underlying source issues.


In [15]:

# 05 — Clarification Compliance
# CL-001: Order identity and KPI restrictions

cl001_checks = pd.DataFrame([
    {
        "check": "Original order IDs preserved",
        "observed": int(df["order_id"].notna().sum()),
        "expected": len(df),
        "status": (
            "PASS" if df["order_id"].notna().all()
            else "FAIL"
        )
    },
    {
        "check": "Order KPIs disabled",
        "observed": int(df["order_kpi_eligible"].sum()),
        "expected": 0,
        "status": (
            "PASS" if not df["order_kpi_eligible"].any()
            else "FAIL"
        )
    },
    {
        "check": "Order keys not certified",
        "observed": int(df["order_key_verified"].sum()),
        "expected": 0,
        "status": (
            "PASS" if not df["order_key_verified"].any()
            else "FAIL"
        )
    }
])

display(cl001_checks)

print(
    "CL-001 preliminary status:",
    "PASS" if cl001_checks["status"].eq("PASS").all()
    else "FAIL"
)


,check,observed,expected,status
0,Original order IDs preserved,23530,23530,PASS
1,Order KPIs disabled,0,0,PASS
2,Order keys not certified,0,0,PASS


CL-001 preliminary status: PASS


In [16]:

# 05 — Clarification Compliance
# CL-001: Independent order identity conflict validation

order_identity = (
    df.groupby("order_id")
    .agg(
        customer_count=("customer_id", "nunique"),
        date_count=("order_date", "nunique")
    )
)

customer_conflicts = int(
    (order_identity["customer_count"] > 1).sum()
)

date_conflicts = int(
    (order_identity["date_count"] > 1).sum()
)

cl001_conflicts = pd.DataFrame([
    {
        "check": "Order IDs with multiple customers",
        "expected": 5518,
        "observed": customer_conflicts
    },
    {
        "check": "Order IDs with multiple dates",
        "expected": 5498,
        "observed": date_conflicts
    }
])

cl001_conflicts["status"] = np.where(
    cl001_conflicts["expected"] == cl001_conflicts["observed"],
    "PASS",
    "FAIL"
)

display(cl001_conflicts)


,check,expected,observed,status
0,Order IDs with multiple customers,5518,5518,PASS
1,Order IDs with multiple dates,5498,5498,PASS


In [17]:

# 05 — Clarification Compliance
# CL-001: Candidate composite key conflict validation

candidate_groups = (
    df.groupby(
        ["order_id", "channel_id", "region_id"],
        dropna=False
    )
    .agg(
        customer_count=("customer_id", "nunique"),
        date_count=("order_date", "nunique"),
        sales_lines=("order_id", "size")
    )
)

conflicting_groups = candidate_groups.loc[
    (candidate_groups["customer_count"] > 1)
    | (candidate_groups["date_count"] > 1)
]

observed_groups = len(conflicting_groups)
observed_lines = int(conflicting_groups["sales_lines"].sum())

cl001_candidate_results = pd.DataFrame([
    {
        "check": "Conflicting candidate groups",
        "expected": 446,
        "observed": observed_groups
    },
    {
        "check": "Sales lines in conflicting groups",
        "expected": 909,
        "observed": observed_lines
    }
])

cl001_candidate_results["status"] = np.where(
    cl001_candidate_results["expected"]
    == cl001_candidate_results["observed"],
    "PASS",
    "FAIL"
)

display(cl001_candidate_results)


,check,expected,observed,status
0,Conflicting candidate groups,446,446,PASS
1,Sales lines in conflicting groups,909,909,PASS


In [18]:

# 05 — Clarification Compliance
# CL-001: Final reporting control validation

cl001_final = pd.DataFrame([
    {
        "check": "All source rows retained",
        "observed": len(df),
        "expected": 23530
    },
    {
        "check": "Unique analytical row keys",
        "observed": df["analytical_row_key"].nunique(),
        "expected": len(df)
    },
    {
        "check": "Verified order keys",
        "observed": int(df["order_key_verified"].sum()),
        "expected": 0
    },
    {
        "check": "Order KPI eligible records",
        "observed": int(df["order_kpi_eligible"].sum()),
        "expected": 0
    }
])

cl001_final["status"] = np.where(
    cl001_final["observed"] == cl001_final["expected"],
    "PASS",
    "FAIL"
)

display(cl001_final)

print(
    "CL-001 reporting controls:",
    "PASS" if cl001_final["status"].eq("PASS").all()
    else "FAIL"
)


,check,observed,expected,status
0,All source rows retained,23530,23530,PASS
1,Unique analytical row keys,23530,23530,PASS
2,Verified order keys,0,0,PASS
3,Order KPI eligible records,0,0,PASS


CL-001 reporting controls: PASS


In [19]:

# 05 — Clarification Compliance
# CL-002: Partial-return exposure validation

partial_returns = df.loc[
    df["order_status"].eq("Partially Returned")
]

cl002_baseline = {
    "record_count": 1075,
    "net_revenue": 673506.50,
    "total_cost": 470229.86,
    "gross_profit": 203276.64
}

cl002_observed = {
    "record_count": len(partial_returns),
    "net_revenue": partial_returns["net_revenue"].sum(),
    "total_cost": partial_returns["total_cost"].sum(),
    "gross_profit": partial_returns["gross_profit"].sum()
}

cl002_results = pd.DataFrame([
    {
        "metric": metric,
        "expected": expected,
        "observed": cl002_observed[metric]
    }
    for metric, expected in cl002_baseline.items()
])

cl002_results["difference"] = (
    cl002_results["observed"] - cl002_results["expected"]
).round(2)

cl002_results["status"] = np.where(
    cl002_results["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

display(cl002_results)

print(
    "CL-002 exposure validation:",
    "PASS" if cl002_results["status"].eq("PASS").all()
    else "FAIL"
)


,metric,expected,observed,difference,status
0,record_count,"1,075.00","1,075.00",0.00,PASS
1,net_revenue,"673,506.50","673,506.50",0.00,PASS
2,total_cost,"470,229.86","470,229.86",0.00,PASS
3,gross_profit,"203,276.64","203,276.64",-0.00,PASS


CL-002 exposure validation: PASS


In [20]:

# 05 — Clarification Compliance
# CL-002: Partial-return source exposure

partial_returns = df.loc[
    df["order_status"].eq("Partially Returned")
]

cl002_baseline = {
    "record_count": 1075,
    "net_revenue": 673506.50,
    "total_cost": 470229.86,
    "gross_profit": 203276.64
}

cl002_observed = {
    "record_count": len(partial_returns),
    "net_revenue": partial_returns["net_revenue"].sum(),
    "total_cost": partial_returns["total_cost"].sum(),
    "gross_profit": partial_returns["gross_profit"].sum()
}

cl002_exposure = pd.DataFrame([
    {
        "metric": metric,
        "expected": expected,
        "observed": cl002_observed[metric]
    }
    for metric, expected in cl002_baseline.items()
])

cl002_exposure["difference"] = (
    cl002_exposure["observed"]
    - cl002_exposure["expected"]
).round(2)

cl002_exposure["status"] = np.where(
    cl002_exposure["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

display(cl002_exposure)


,metric,expected,observed,difference,status
0,record_count,"1,075.00","1,075.00",0.00,PASS
1,net_revenue,"673,506.50","673,506.50",0.00,PASS
2,total_cost,"470,229.86","470,229.86",0.00,PASS
3,gross_profit,"203,276.64","203,276.64",-0.00,PASS


In [21]:
# 05 — Clarification Compliance
# CL-002: Independent population eligibility

cl002_expected_certified = (
    df["order_status"].eq("Completed")
    & ~expected_invalid_numeric
)

cl002_expected_provisional = (
    df["order_status"].isin(
        ["Completed", "Partially Returned"]
    )
    & ~expected_invalid_numeric
)

cl002_populations = pd.DataFrame([
    {
        "population": "Certified",
        "expected": 21342,
        "observed": int(cl002_expected_certified.sum()),
        "flag_mismatches": int(
            (
                cl002_expected_certified
                != df["certified_sales_eligible"]
            ).sum()
        )
    },
    {
        "population": "Provisional",
        "expected": 22417,
        "observed": int(cl002_expected_provisional.sum()),
        "flag_mismatches": int(
            (
                cl002_expected_provisional
                != df["provisional_return_inclusive_eligible"]
            ).sum()
        )
    }
])

cl002_populations["status"] = np.where(
    (cl002_populations["expected"] == cl002_populations["observed"])
    & (cl002_populations["flag_mismatches"] == 0),
    "PASS",
    "FAIL"
)

display(cl002_populations)


,population,expected,observed,flag_mismatches,status
0,Certified,21342,21342,0,PASS
1,Provisional,22417,22417,0,PASS


In [22]:
# 05 — Clarification Compliance
# CL-002: Excluded transaction statuses

excluded_statuses = df.loc[
    df["order_status"].isin(["Pending", "Cancelled"])
]

incorrectly_eligible = (
    excluded_statuses["certified_sales_eligible"]
    | excluded_statuses["provisional_return_inclusive_eligible"]
).sum()

cl002_exclusions = pd.DataFrame([
    {
        "check": "Pending + Cancelled records",
        "expected": 1110,
        "observed": len(excluded_statuses)
    },
    {
        "check": "Incorrectly eligible records",
        "expected": 0,
        "observed": int(incorrectly_eligible)
    }
])

cl002_exclusions["status"] = np.where(
    cl002_exclusions["expected"]
    == cl002_exclusions["observed"],
    "PASS",
    "FAIL"
)

display(cl002_exclusions)


,check,expected,observed,status
0,Pending + Cancelled records,1110,1110,PASS
1,Incorrectly eligible records,0,0,PASS


In [23]:
# 05 — Clarification Compliance
# CL-002: Population relationship validation

certified_mask = df["certified_sales_eligible"]
provisional_mask = df["provisional_return_inclusive_eligible"]

certified_outside_provisional = int(
    (certified_mask & ~provisional_mask).sum()
)

provisional_only_mask = provisional_mask & ~certified_mask

provisional_only_count = int(provisional_only_mask.sum())

unexpected_provisional_only = int(
    (
        provisional_only_mask
        & ~df["order_status"].eq("Partially Returned")
    ).sum()
)

cl002_relationship = pd.DataFrame([
    {
        "check": "Certified outside provisional",
        "expected": 0,
        "observed": certified_outside_provisional
    },
    {
        "check": "Provisional-only records",
        "expected": 1075,
        "observed": provisional_only_count
    },
    {
        "check": "Unexpected provisional-only statuses",
        "expected": 0,
        "observed": unexpected_provisional_only
    }
])

cl002_relationship["status"] = np.where(
    cl002_relationship["expected"]
    == cl002_relationship["observed"],
    "PASS",
    "FAIL"
)

display(cl002_relationship)


,check,expected,observed,status
0,Certified outside provisional,0,0,PASS
1,Provisional-only records,1075,1075,PASS
2,Unexpected provisional-only statuses,0,0,PASS


In [24]:
# 05 — Clarification Compliance
# CL-002: Provisional-only financial reconciliation

financial_metrics = [
    "net_revenue",
    "total_cost",
    "gross_profit"
]

provisional_increment = (
    df.loc[provisional_only_mask, financial_metrics]
    .sum()
)

cl002_financial_bridge = pd.DataFrame([
    {
        "metric": metric,
        "expected": cl002_baseline[metric],
        "observed": provisional_increment[metric]
    }
    for metric in financial_metrics
])

cl002_financial_bridge["difference"] = (
    cl002_financial_bridge["observed"]
    - cl002_financial_bridge["expected"]
).round(2)

cl002_financial_bridge["status"] = np.where(
    cl002_financial_bridge["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

display(cl002_financial_bridge)


,metric,expected,observed,difference,status
0,net_revenue,"673,506.50","673,506.50",0.00,PASS
1,total_cost,"470,229.86","470,229.86",0.00,PASS
2,gross_profit,"203,276.64","203,276.64",-0.00,PASS


In [25]:

# 05 — Clarification Compliance
# CL002 validation evidence summary

cl002_evidence_groups = {
    "Financial population reconciliation": cl002_results,
    "Partial-return financial exposure": cl002_exposure,
    "Population eligibility": cl002_populations,
    "Exclusion compliance": cl002_exclusions,
    "Population relationship": cl002_relationship,
    "Financial reconciliation bridge": cl002_financial_bridge
}

cl002_summary = pd.DataFrame([
    {
        "control": name,
        "checks": len(evidence),
        "passed": int(evidence["status"].eq("PASS").sum()),
        "failed": int(evidence["status"].ne("PASS").sum()),
        "status": (
            "PASS"
            if len(evidence) > 0
            and evidence["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for name, evidence in cl002_evidence_groups.items()
])

display(cl002_summary)

print(
    "\nCL002 validation summary:",
    "PASS"
    if len(cl002_summary) == 6
    and cl002_summary["status"].eq("PASS").all()
    else "FAIL"
)


,control,checks,passed,failed,status
0,Financial population reconciliation,4,4,0,PASS
1,Partial-return financial exposure,4,4,0,PASS
2,Population eligibility,2,2,0,PASS
3,Exclusion compliance,2,2,0,PASS
4,Population relationship,3,3,0,PASS
5,Financial reconciliation bridge,3,3,0,PASS



CL002 validation summary: PASS


In [26]:

# 05 — Clarification Compliance
# CL-003: Affected transaction identification

cl003_record = df.loc[
    df["sales_line_id"].eq("SL0020778")
]

cl003_fields = [
    "sales_line_id",
    "order_id",
    "payment_method_source",
    "payment_method_reporting"
]

display(cl003_record[cl003_fields])

print("Matching records:", len(cl003_record))


,sales_line_id,order_id,payment_method_source,payment_method_reporting
20777,SL0020778,O0013076,Credit Young Card,Unmapped


Matching records: 1


In [27]:

# 05 — Clarification Compliance
# CL-003: Approved payment classification validation

cl003_checks = {
    "Record identified": len(cl003_record) == 1,
    "Order ID preserved": (
        cl003_record["order_id"].eq("O0013076").all()
    ),
    "Source payment preserved": (
        cl003_record["payment_method_source"]
        .eq("Credit Young Card").all()
    ),
    "Reporting classification": (
        cl003_record["payment_method_reporting"]
        .eq("Unmapped").all()
    )
}

cl003_classification = pd.DataFrame([
    {
        "check": check,
        "status": "PASS" if passed else "FAIL"
    }
    for check, passed in cl003_checks.items()
])

display(cl003_classification)


,check,status
0,Record identified,PASS
1,Order ID preserved,PASS
2,Source payment preserved,PASS
3,Reporting classification,PASS


In [28]:

# 05 — Clarification Compliance
# CL-003: Source-to-reporting mapping validation

cl003_source_mask = df[
    "payment_method_source"
].eq("Credit Young Card")

cl003_mapping_errors = (
    cl003_source_mask
    & ~df["payment_method_reporting"].eq("Unmapped")
)

cl003_mapping = pd.DataFrame([
    {
        "check": "Credit Young Card source records",
        "expected": 1,
        "observed": int(cl003_source_mask.sum())
    },
    {
        "check": "Incorrect reporting classifications",
        "expected": 0,
        "observed": int(cl003_mapping_errors.sum())
    }
])

cl003_mapping["status"] = np.where(
    cl003_mapping["expected"]
    == cl003_mapping["observed"],
    "PASS",
    "FAIL"
)

display(cl003_mapping)


,check,expected,observed,status
0,Credit Young Card source records,1,1,PASS
1,Incorrect reporting classifications,0,0,PASS


In [29]:

# 05 — Clarification Compliance
# CL-003: Financial eligibility independence

cl003_expected_certified = (
    cl003_record["order_status"].eq("Completed")
    & ~cl003_record["invalid_numeric_flag"]
)

cl003_expected_provisional = (
    cl003_record["order_status"].isin(
        ["Completed", "Partially Returned"]
    )
    & ~cl003_record["invalid_numeric_flag"]
)

cl003_eligibility = pd.DataFrame([
    {
        "check": "Certified eligibility",
        "expected": int(cl003_expected_certified.sum()),
        "observed": int(
            cl003_record["certified_sales_eligible"].sum()
        )
    },
    {
        "check": "Provisional eligibility",
        "expected": int(cl003_expected_provisional.sum()),
        "observed": int(
            cl003_record[
                "provisional_return_inclusive_eligible"
            ].sum()
        )
    }
])

cl003_eligibility["status"] = np.where(
    cl003_eligibility["expected"]
    == cl003_eligibility["observed"],
    "PASS",
    "FAIL"
)

display(cl003_eligibility)


,check,expected,observed,status
0,Certified eligibility,1,1,PASS
1,Provisional eligibility,1,1,PASS


In [30]:

# 05 — Clarification Compliance
# CL-003: Compliance validation summary

cl003_summary = pd.DataFrame([
    {
        "control": "Affected transaction identification",
        "status": (
            "PASS" if len(cl003_record) == 1 else "FAIL"
        )
    },
    {
        "control": "Approved payment classification",
        "status": (
            "PASS"
            if cl003_classification["status"].eq("PASS").all()
            else "FAIL"
        )
    },
    {
        "control": "Source-to-reporting mapping",
        "status": (
            "PASS"
            if cl003_mapping["status"].eq("PASS").all()
            else "FAIL"
        )
    },
    {
        "control": "Financial eligibility independence",
        "status": (
            "PASS"
            if cl003_eligibility["status"].eq("PASS").all()
            else "FAIL"
        )
    }
])

display(cl003_summary)

print(
    "CL-003 analytical implementation:",
    "PASS"
    if cl003_summary["status"].eq("PASS").all()
    else "FAIL"
)


,control,status
0,Affected transaction identification,PASS
1,Approved payment classification,PASS
2,Source-to-reporting mapping,PASS
3,Financial eligibility independence,PASS


CL-003 analytical implementation: PASS


In [31]:

# 05 — Clarification Compliance
# CL-004: Duplicate sales line identification

cl004_expected_ids = [
    "SL0000251",
    "SL0008001",
    "SL0017001"
]

cl004_duplicates = (
    df.loc[
        df["sales_line_id"].isin(cl004_expected_ids)
    ]
    .groupby("sales_line_id")
    .size()
    .reindex(cl004_expected_ids, fill_value=0)
)

cl004_duplicate_results = pd.DataFrame({
    "sales_line_id": cl004_expected_ids,
    "expected": [2, 2, 2],
    "observed": cl004_duplicates.to_numpy()
})

cl004_duplicate_results["status"] = np.where(
    cl004_duplicate_results["expected"]
    == cl004_duplicate_results["observed"],
    "PASS",
    "FAIL"
)

display(cl004_duplicate_results)


,sales_line_id,expected,observed,status
0,SL0000251,2,2,PASS
1,SL0008001,2,2,PASS
2,SL0017001,2,2,PASS


In [32]:

# 05 — Clarification Compliance
# CL-004: Complete sales line collision assessment

cl004_all_collisions = (
    df["sales_line_id"]
    .value_counts(dropna=False)
)

cl004_all_collisions = cl004_all_collisions.loc[
    cl004_all_collisions > 1
]

cl004_collision_results = pd.DataFrame([
    {
        "check": "Distinct colliding IDs",
        "expected": 3,
        "observed": len(cl004_all_collisions)
    },
    {
        "check": "Rows affected by collisions",
        "expected": 6,
        "observed": int(cl004_all_collisions.sum())
    },
    {
        "check": "Undocumented colliding IDs",
        "expected": 0,
        "observed": len(
            set(cl004_all_collisions.index)
            - set(cl004_expected_ids)
        )
    }
])

cl004_collision_results["status"] = np.where(
    cl004_collision_results["expected"]
    == cl004_collision_results["observed"],
    "PASS",
    "FAIL"
)

display(cl004_collision_results)


,check,expected,observed,status
0,Distinct colliding IDs,3,3,PASS
1,Rows affected by collisions,6,6,PASS
2,Undocumented colliding IDs,0,0,PASS


In [33]:

# 05 — Clarification Compliance
# CL-004: Collision flag accuracy validation

cl004_expected_collision_flag = (
    df["sales_line_id"].duplicated(keep=False)
)

cl004_flag_mismatches = (
    cl004_expected_collision_flag
    != df["line_id_collision_flag"]
)

cl004_flag_results = pd.DataFrame([
    {
        "check": "Expected collision-flagged rows",
        "expected": 6,
        "observed": int(
            df["line_id_collision_flag"].sum()
        )
    },
    {
        "check": "Collision flag mismatches",
        "expected": 0,
        "observed": int(
            cl004_flag_mismatches.sum()
        )
    }
])

cl004_flag_results["status"] = np.where(
    cl004_flag_results["expected"]
    == cl004_flag_results["observed"],
    "PASS",
    "FAIL"
)

display(cl004_flag_results)


,check,expected,observed,status
0,Expected collision-flagged rows,6,6,PASS
1,Collision flag mismatches,0,0,PASS


In [34]:

# 05 — Clarification Compliance
# CL-004: Original sales line identifier preservation

cl004_source_mismatches = (
    df["sales_line_id"].fillna("<NULL>")
    != df["sales_line_id_source"].fillna("<NULL>")
)

cl004_source_results = pd.DataFrame([
    {
        "check": "Missing source identifiers",
        "expected": 0,
        "observed": int(
            df["sales_line_id_source"].isna().sum()
        )
    },
    {
        "check": "Source identifier mismatches",
        "expected": 0,
        "observed": int(
            cl004_source_mismatches.sum()
        )
    }
])

cl004_source_results["status"] = np.where(
    cl004_source_results["expected"]
    == cl004_source_results["observed"],
    "PASS",
    "FAIL"
)

display(cl004_source_results)


,check,expected,observed,status
0,Missing source identifiers,0,0,PASS
1,Source identifier mismatches,0,0,PASS


In [35]:

# 05 — Clarification Compliance
# CL-004: Analytical row key uniqueness validation

cl004_key_results = pd.DataFrame([
    {
        "check": "Total analytical rows",
        "expected": 23530,
        "observed": len(df)
    },
    {
        "check": "Missing analytical row keys",
        "expected": 0,
        "observed": int(
            df["analytical_row_key"].isna().sum()
        )
    },
    {
        "check": "Duplicate analytical row keys",
        "expected": 0,
        "observed": int(
            df["analytical_row_key"].duplicated().sum()
        )
    },
    {
        "check": "Distinct keys among collision rows",
        "expected": 6,
        "observed": int(
            df.loc[
                cl004_expected_collision_flag,
                "analytical_row_key"
            ].nunique()
        )
    }
])

cl004_key_results["status"] = np.where(
    cl004_key_results["expected"]
    == cl004_key_results["observed"],
    "PASS",
    "FAIL"
)

display(cl004_key_results)


,check,expected,observed,status
0,Total analytical rows,23530,23530,PASS
1,Missing analytical row keys,0,0,PASS
2,Duplicate analytical row keys,0,0,PASS
3,Distinct keys among collision rows,6,6,PASS


In [36]:

# 05 — Clarification Compliance
# CL-004: Collision-independent financial eligibility

cl004_affected = df.loc[
    cl004_expected_collision_flag
]

cl004_expected_certified = (
    cl004_affected["order_status"].eq("Completed")
    & ~cl004_affected["invalid_numeric_flag"]
)

cl004_expected_provisional = (
    cl004_affected["order_status"].isin(
        ["Completed", "Partially Returned"]
    )
    & ~cl004_affected["invalid_numeric_flag"]
)

cl004_eligibility_results = pd.DataFrame([
    {
        "check": "Certified eligibility mismatches",
        "expected": 0,
        "observed": int(
            (
                cl004_expected_certified
                != cl004_affected[
                    "certified_sales_eligible"
                ]
            ).sum()
        )
    },
    {
        "check": "Provisional eligibility mismatches",
        "expected": 0,
        "observed": int(
            (
                cl004_expected_provisional
                != cl004_affected[
                    "provisional_return_inclusive_eligible"
                ]
            ).sum()
        )
    }
])

cl004_eligibility_results["status"] = np.where(
    cl004_eligibility_results["expected"]
    == cl004_eligibility_results["observed"],
    "PASS",
    "FAIL"
)

display(cl004_eligibility_results)


,check,expected,observed,status
0,Certified eligibility mismatches,0,0,PASS
1,Provisional eligibility mismatches,0,0,PASS


In [37]:

# 05 — Clarification Compliance
# CL-004: Compliance validation summary

cl004_control_groups = {
    "Documented collision IDs": cl004_duplicate_results,
    "Complete collision assessment": cl004_collision_results,
    "Collision flag accuracy": cl004_flag_results,
    "Source identifier consistency": cl004_source_results,
    "Analytical key uniqueness": cl004_key_results,
    "Financial eligibility independence": cl004_eligibility_results
}

cl004_summary = pd.DataFrame([
    {
        "control": control,
        "status": (
            "PASS"
            if results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for control, results in cl004_control_groups.items()
])

display(cl004_summary)

print(
    "CL-004 analytical implementation:",
    "PASS"
    if cl004_summary["status"].eq("PASS").all()
    else "FAIL"
)


,control,status
0,Documented collision IDs,PASS
1,Complete collision assessment,PASS
2,Collision flag accuracy,PASS
3,Source identifier consistency,PASS
4,Analytical key uniqueness,PASS
5,Financial eligibility independence,PASS


CL-004 analytical implementation: PASS


In [38]:

# 05 — Clarification Compliance
# CL-005: Invalid numeric record identification

cl005_expected_ids = [
    "SL0012001",
    "SL0015101",
    "SL0018301"
]

cl005_fields = [
    "sales_line_id",
    "order_status",
    "quantity",
    "unit_price",
    "unit_cost",
    "discount_pct",
    "invalid_numeric_flag"
]

cl005_affected = df.loc[
    df["sales_line_id"].isin(cl005_expected_ids),
    cl005_fields
].copy()

display(cl005_affected.sort_values("sales_line_id"))

print("Affected records:", len(cl005_affected))


,sales_line_id,order_status,quantity,unit_price,unit_cost,discount_pct,invalid_numeric_flag
12000,SL0012001,Completed,-2,502.70,330.31,0.00,True
15100,SL0015101,Completed,1,626.96,-4.75,0.10,True
18300,SL0018301,Completed,2,319.04,199.58,1.15,True


Affected records: 3


In [39]:

# 05 — Clarification Compliance
# CL-005: Documented numeric defect validation

cl005_expected_defects = {
    "SL0012001": ("quantity", -2),
    "SL0015101": ("unit_cost", -4.75),
    "SL0018301": ("discount_pct", 1.15)
}

cl005_defect_results = []

for sales_line_id, (field, expected_value) in cl005_expected_defects.items():

    matching = df.loc[
        df["sales_line_id"].eq(sales_line_id)
    ]

    record_found = len(matching) == 1

    value_matches = (
        record_found
        and np.isclose(
            matching[field].iloc[0],
            expected_value,
            atol=0.000001
        )
    )

    flag_matches = (
        record_found
        and bool(matching["invalid_numeric_flag"].iloc[0])
    )

    cl005_defect_results.append({
        "sales_line_id": sales_line_id,
        "field": field,
        "expected_value": expected_value,
        "observed_value": (
            matching[field].iloc[0] if record_found else np.nan
        ),
        "status": (
            "PASS"
            if record_found and value_matches and flag_matches
            else "FAIL"
        )
    })

cl005_defect_results = pd.DataFrame(cl005_defect_results)

display(cl005_defect_results)


,sales_line_id,field,expected_value,observed_value,status
0,SL0012001,quantity,-2.00,-2.00,PASS
1,SL0015101,unit_cost,-4.75,-4.75,PASS
2,SL0018301,discount_pct,1.15,1.15,PASS


In [40]:

# 05 — Clarification Compliance
# CL-005: Independent numeric validity reconstruction

cl005_numeric_fields = [
    "quantity",
    "unit_price",
    "unit_cost",
    "discount_pct"
]

# Reconstruct approved numeric validity rules
cl005_expected_invalid = (
    df[cl005_numeric_fields].isna().any(axis=1)
    | df["quantity"].le(0)
    | df["unit_price"].le(0)
    | df["unit_cost"].lt(0)
    | ~df["discount_pct"].between(0, 1)
)

# Compare reconstructed results against existing flags
cl005_flag_mismatches = (
    cl005_expected_invalid
    != df["invalid_numeric_flag"]
)

# Validate expected findings
cl005_numeric_results = pd.DataFrame([
    {
        "check": "Invalid numeric records",
        "expected": 3,
        "observed": int(cl005_expected_invalid.sum())
    },
    {
        "check": "Invalid numeric flag mismatches",
        "expected": 0,
        "observed": int(cl005_flag_mismatches.sum())
    },
    {
        "check": "Undocumented invalid records",
        "expected": 0,
        "observed": int(
            (
                cl005_expected_invalid
                & ~df["sales_line_id"].isin(cl005_expected_ids)
            ).sum()
        )
    }
])

cl005_numeric_results["status"] = np.where(
    cl005_numeric_results["expected"]
    == cl005_numeric_results["observed"],
    "PASS",
    "FAIL"
)

display(cl005_numeric_results)

print(
    "\nCL-005 numeric validation:",
    "PASS"
    if cl005_numeric_results["status"].eq("PASS").all()
    else "FAIL"
)

print(
    "Variable available:",
    "cl005_expected_invalid" in globals()
)


,check,expected,observed,status
0,Invalid numeric records,3,3,PASS
1,Invalid numeric flag mismatches,0,0,PASS
2,Undocumented invalid records,0,0,PASS



CL-005 numeric validation: PASS
Variable available: True


In [41]:

# 05 — Clarification Compliance
# CL-005: Invalid record financial exclusion validation

cl005_invalid_rows = df.loc[
    cl005_expected_invalid
]

cl005_exclusion_results = pd.DataFrame([
    {
        "check": "Invalid records retained",
        "expected": 3,
        "observed": len(cl005_invalid_rows)
    },
    {
        "check": "Invalid records in certified population",
        "expected": 0,
        "observed": int(
            cl005_invalid_rows["certified_sales_eligible"].sum()
        )
    },
    {
        "check": "Invalid records in provisional population",
        "expected": 0,
        "observed": int(
            cl005_invalid_rows[
                "provisional_return_inclusive_eligible"
            ].sum()
        )
    }
])

cl005_exclusion_results["status"] = np.where(
    cl005_exclusion_results["expected"]
    == cl005_exclusion_results["observed"],
    "PASS",
    "FAIL"
)

display(cl005_exclusion_results)


,check,expected,observed,status
0,Invalid records retained,3,3,PASS
1,Invalid records in certified population,0,0,PASS
2,Invalid records in provisional population,0,0,PASS


In [42]:

# 05 — Clarification Compliance
# CL-005: Original defect retention validation

cl005_retention_results = pd.DataFrame([
    {
        "check": "Negative quantity retained",
        "expected": 1,
        "observed": int(
            (
                df["sales_line_id"].eq("SL0012001")
                & df["quantity"].eq(-2)
            ).sum()
        )
    },
    {
        "check": "Negative unit cost retained",
        "expected": 1,
        "observed": int(
            (
                df["sales_line_id"].eq("SL0015101")
                & np.isclose(df["unit_cost"], -4.75)
            ).sum()
        )
    },
    {
        "check": "Out-of-range discount retained",
        "expected": 1,
        "observed": int(
            (
                df["sales_line_id"].eq("SL0018301")
                & np.isclose(df["discount_pct"], 1.15)
            ).sum()
        )
    }
])

cl005_retention_results["status"] = np.where(
    cl005_retention_results["expected"]
    == cl005_retention_results["observed"],
    "PASS",
    "FAIL"
)

display(cl005_retention_results)


,check,expected,observed,status
0,Negative quantity retained,1,1,PASS
1,Negative unit cost retained,1,1,PASS
2,Out-of-range discount retained,1,1,PASS


In [43]:

# 05 — Clarification Compliance
# CL-005: Compliance validation summary

cl005_control_groups = {
    "Documented numeric defects": cl005_defect_results,
    "Independent numeric validation": cl005_numeric_results,
    "Financial exclusion": cl005_exclusion_results,
    "Original defect retention": cl005_retention_results
}

cl005_summary = pd.DataFrame([
    {
        "control": control,
        "status": (
            "PASS"
            if results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for control, results in cl005_control_groups.items()
])

display(cl005_summary)

print(
    "CL-005 analytical implementation:",
    "PASS"
    if cl005_summary["status"].eq("PASS").all()
    else "FAIL"
)


,control,status
0,Documented numeric defects,PASS
1,Independent numeric validation,PASS
2,Financial exclusion,PASS
3,Original defect retention,PASS


CL-005 analytical implementation: PASS


In [44]:

# 05 — Clarification Compliance
# CL-006: Missing reference record identification

cl006_expected_ids = [
    "SL0000105",
    "SL0008422"
]

cl006_fields = [
    "sales_line_id",
    "customer_id",
    "channel_id",
    "customer_attribution",
    "channel_attribution"
]

cl006_affected = df.loc[
    df["sales_line_id"].isin(cl006_expected_ids),
    cl006_fields
].copy()

display(cl006_affected.sort_values("sales_line_id"))

print("Affected records:", len(cl006_affected))


,sales_line_id,customer_id,channel_id,customer_attribution,channel_attribution
104,SL0000105,NaN,CH04,Unknown Customer,CH04
8421,SL0008422,C02226,NaN,C02226,Unknown Channel


Affected records: 2


In [45]:

# 05 — Clarification Compliance
# CL-006: Original missing reference preservation

cl006_reference_checks = {
    "Missing customer reference preserved": (
        len(df.loc[df["sales_line_id"].eq("SL0000105")]) == 1
        and df.loc[
            df["sales_line_id"].eq("SL0000105"),
            "customer_id"
        ].isna().all()
    ),
    "Missing channel reference preserved": (
        len(df.loc[df["sales_line_id"].eq("SL0008422")]) == 1
        and df.loc[
            df["sales_line_id"].eq("SL0008422"),
            "channel_id"
        ].isna().all()
    )
}

cl006_reference_results = pd.DataFrame([
    {
        "check": check,
        "status": "PASS" if passed else "FAIL"
    }
    for check, passed in cl006_reference_checks.items()
])

display(cl006_reference_results)


,check,status
0,Missing customer reference preserved,PASS
1,Missing channel reference preserved,PASS


In [46]:

# 05 — Clarification Compliance
# CL-006: Approved attribution classification validation

cl006_attribution_checks = {
    "Unknown Customer attribution": (
        df.loc[
            df["sales_line_id"].eq("SL0000105"),
            "customer_attribution"
        ].eq("Unknown Customer").all()
    ),
    "Unknown Channel attribution": (
        df.loc[
            df["sales_line_id"].eq("SL0008422"),
            "channel_attribution"
        ].eq("Unknown Channel").all()
    )
}

cl006_attribution_results = pd.DataFrame([
    {
        "check": check,
        "status": "PASS" if passed else "FAIL"
    }
    for check, passed in cl006_attribution_checks.items()
])

display(cl006_attribution_results)


,check,status
0,Unknown Customer attribution,PASS
1,Unknown Channel attribution,PASS


In [47]:

# 05 — Clarification Compliance
# CL-006: Missing reference coverage validation

cl006_missing_customer = df["customer_id"].isna()
cl006_missing_channel = df["channel_id"].isna()

cl006_coverage_results = pd.DataFrame([
    {
        "check": "Missing customer references",
        "expected": 1,
        "observed": int(cl006_missing_customer.sum())
    },
    {
        "check": "Missing channel references",
        "expected": 1,
        "observed": int(cl006_missing_channel.sum())
    },
    {
        "check": "Undocumented missing customer references",
        "expected": 0,
        "observed": int(
            (
                cl006_missing_customer
                & ~df["sales_line_id"].eq("SL0000105")
            ).sum()
        )
    },
    {
        "check": "Undocumented missing channel references",
        "expected": 0,
        "observed": int(
            (
                cl006_missing_channel
                & ~df["sales_line_id"].eq("SL0008422")
            ).sum()
        )
    }
])

cl006_coverage_results["status"] = np.where(
    cl006_coverage_results["expected"]
    == cl006_coverage_results["observed"],
    "PASS",
    "FAIL"
)

display(cl006_coverage_results)


,check,expected,observed,status
0,Missing customer references,1,1,PASS
1,Missing channel references,1,1,PASS
2,Undocumented missing customer references,0,0,PASS
3,Undocumented missing channel references,0,0,PASS


In [48]:

# 05 — Clarification Compliance
# CL-006: Missing reference financial eligibility independence

cl006_financial_rows = df.loc[
    df["sales_line_id"].isin(cl006_expected_ids)
]

cl006_expected_certified = (
    cl006_financial_rows["order_status"].eq("Completed")
    & ~cl006_financial_rows["invalid_numeric_flag"]
)

cl006_expected_provisional = (
    cl006_financial_rows["order_status"].isin(
        ["Completed", "Partially Returned"]
    )
    & ~cl006_financial_rows["invalid_numeric_flag"]
)

cl006_eligibility_results = pd.DataFrame([
    {
        "check": "Certified eligibility mismatches",
        "expected": 0,
        "observed": int(
            (
                cl006_expected_certified
                != cl006_financial_rows[
                    "certified_sales_eligible"
                ]
            ).sum()
        )
    },
    {
        "check": "Provisional eligibility mismatches",
        "expected": 0,
        "observed": int(
            (
                cl006_expected_provisional
                != cl006_financial_rows[
                    "provisional_return_inclusive_eligible"
                ]
            ).sum()
        )
    }
])

cl006_eligibility_results["status"] = np.where(
    cl006_eligibility_results["expected"]
    == cl006_eligibility_results["observed"],
    "PASS",
    "FAIL"
)

display(cl006_eligibility_results)


,check,expected,observed,status
0,Certified eligibility mismatches,0,0,PASS
1,Provisional eligibility mismatches,0,0,PASS


In [49]:

# 05 — Clarification Compliance
# CL-006: Unknown attribution aggregation retention

cl006_certified = df.loc[
    df["certified_sales_eligible"]
]

cl006_total_revenue = cl006_certified["net_revenue"].sum()

cl006_customer_revenue = (
    cl006_certified
    .groupby("customer_attribution", dropna=False)["net_revenue"]
    .sum()
)

cl006_channel_revenue = (
    cl006_certified
    .groupby("channel_attribution", dropna=False)["net_revenue"]
    .sum()
)

cl006_aggregation_results = pd.DataFrame([
    {
        "check": "Customer attribution revenue reconciliation",
        "difference": round(
            cl006_customer_revenue.sum() - cl006_total_revenue,
            2
        )
    },
    {
        "check": "Channel attribution revenue reconciliation",
        "difference": round(
            cl006_channel_revenue.sum() - cl006_total_revenue,
            2
        )
    }
])

cl006_aggregation_results["status"] = np.where(
    cl006_aggregation_results["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

display(cl006_aggregation_results)


,check,difference,status
0,Customer attribution revenue reconciliation,0.00,PASS
1,Channel attribution revenue reconciliation,0.00,PASS


In [50]:

# 05 — Clarification Compliance
# CL-006: Compliance validation summary

cl006_control_groups = {
    "Original missing reference preservation": cl006_reference_results,
    "Approved attribution classification": cl006_attribution_results,
    "Missing reference coverage": cl006_coverage_results,
    "Financial eligibility independence": cl006_eligibility_results,
    "Aggregation retention": cl006_aggregation_results
}

cl006_summary = pd.DataFrame([
    {
        "control": control,
        "status": (
            "PASS"
            if results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for control, results in cl006_control_groups.items()
])

display(cl006_summary)

print(
    "CL-006 analytical implementation:",
    "PASS"
    if cl006_summary["status"].eq("PASS").all()
    else "FAIL"
)


,control,status
0,Original missing reference preservation,PASS
1,Approved attribution classification,PASS
2,Missing reference coverage,PASS
3,Financial eligibility independence,PASS
4,Aggregation retention,PASS


CL-006 analytical implementation: PASS


In [51]:

# 05 — Clarification Compliance
# CL-007: Original outlier and negative profit exposure

cl007_high_cost = df.loc[
    df["high_total_cost_outlier_flag"]
    & df["total_cost"].gt(0)
]

cl007_negative_profit = df.loc[
    df["negative_gross_profit_flag"]
]

cl007_high_cost_negative_profit = df.loc[
    df["high_cost_negative_profit_flag"]
]

cl007_exposure_results = pd.DataFrame([
    {
        "check": "Positive high-cost outlier records",
        "expected": 2610,
        "observed": len(cl007_high_cost)
    },
    {
        "check": "All negative gross profit records",
        "expected": 1735,
        "observed": len(cl007_negative_profit)
    },
    {
        "check": "All negative gross profit amount",
        "expected": -62241.49,
        "observed": round(
            cl007_negative_profit["gross_profit"].sum(), 2
        )
    },
    {
        "check": "High-cost and negative-profit records",
        "expected": 190,
        "observed": len(cl007_high_cost_negative_profit)
    },
    {
        "check": "High-cost and negative-profit amount",
        "expected": -23412.18,
        "observed": round(
            cl007_high_cost_negative_profit["gross_profit"].sum(), 2
        )
    }
])

cl007_exposure_results["difference"] = (
    cl007_exposure_results["observed"]
    - cl007_exposure_results["expected"]
).round(2)

cl007_exposure_results["status"] = np.where(
    cl007_exposure_results["difference"].abs() < 0.005,
    "PASS",
    "FAIL"
)

display(cl007_exposure_results)


,check,expected,observed,difference,status
0,Positive high-cost outlier records,"2,610.00","2,610.00",0.00,PASS
1,All negative gross profit records,"1,735.00","1,735.00",0.00,PASS
2,All negative gross profit amount,"-62,241.49","-62,241.49",0.00,PASS
3,High-cost and negative-profit records,190.00,190.00,0.00,PASS
4,High-cost and negative-profit amount,"-23,412.18","-23,412.18",0.00,PASS


In [52]:

# 05 — Clarification Compliance
# CL-007: Negative gross profit flag validation

cl007_expected_negative_flag = (
    df["gross_profit"].lt(0)
)

cl007_expected_combined_flag = (
    df["high_total_cost_outlier_flag"]
    & cl007_expected_negative_flag
)

cl007_negative_flag_results = pd.DataFrame([
    {
        "check": "Negative gross profit flag mismatches",
        "expected": 0,
        "observed": int(
            (
                cl007_expected_negative_flag
                != df["negative_gross_profit_flag"]
            ).sum()
        )
    },
    {
        "check": "Combined anomaly flag mismatches",
        "expected": 0,
        "observed": int(
            (
                cl007_expected_combined_flag
                != df["high_cost_negative_profit_flag"]
            ).sum()
        )
    }
])

cl007_negative_flag_results["status"] = np.where(
    cl007_negative_flag_results["expected"]
    == cl007_negative_flag_results["observed"],
    "PASS",
    "FAIL"
)

display(cl007_negative_flag_results)


,check,expected,observed,status
0,Negative gross profit flag mismatches,0,0,PASS
1,Combined anomaly flag mismatches,0,0,PASS


In [53]:

# 05 — Clarification Compliance
# CL-007: Total cost IQR methodology inspection

cl007_positive_costs = df.loc[
    df["total_cost"].gt(0),
    "total_cost"
]

cl007_q1 = cl007_positive_costs.quantile(0.25)
cl007_q3 = cl007_positive_costs.quantile(0.75)
cl007_iqr = cl007_q3 - cl007_q1

cl007_lower_fence = cl007_q1 - 1.5 * cl007_iqr
cl007_upper_fence = cl007_q3 + 1.5 * cl007_iqr

cl007_expected_high_cost = (
    df["total_cost"].gt(cl007_upper_fence)
)

cl007_iqr_inspection = pd.DataFrame([
    {
        "metric": "Positive cost records",
        "value": len(cl007_positive_costs)
    },
    {
        "metric": "Q1",
        "value": cl007_q1
    },
    {
        "metric": "Q3",
        "value": cl007_q3
    },
    {
        "metric": "IQR",
        "value": cl007_iqr
    },
    {
        "metric": "Lower fence",
        "value": cl007_lower_fence
    },
    {
        "metric": "Upper fence",
        "value": cl007_upper_fence
    },
    {
        "metric": "Reconstructed high-cost records",
        "value": int(cl007_expected_high_cost.sum())
    },
    {
        "metric": "Existing high-cost flagged records",
        "value": int(
            df["high_total_cost_outlier_flag"].sum()
        )
    }
])

display(cl007_iqr_inspection.round(2))


,metric,value
0,Positive cost records,"23,528.00"
1,Q1,153.78
2,Q3,482.01
3,IQR,328.23
4,Lower fence,-338.57
5,Upper fence,974.36
6,Reconstructed high-cost records,"2,610.00"
7,Existing high-cost flagged records,"2,610.00"


In [54]:

# 05 — Clarification Compliance
# CL-007: Independent IQR flag accuracy validation

cl007_iqr_mismatches = (
    cl007_expected_high_cost
    != df["high_total_cost_outlier_flag"]
)

cl007_iqr_results = pd.DataFrame([
    {
        "check": "Reconstructed high-cost records",
        "expected": 2610,
        "observed": int(cl007_expected_high_cost.sum())
    },
    {
        "check": "Existing high-cost flagged records",
        "expected": 2610,
        "observed": int(
            df["high_total_cost_outlier_flag"].sum()
        )
    },
    {
        "check": "IQR flag mismatches",
        "expected": 0,
        "observed": int(cl007_iqr_mismatches.sum())
    }
])

cl007_iqr_results["status"] = np.where(
    cl007_iqr_results["expected"]
    == cl007_iqr_results["observed"],
    "PASS",
    "FAIL"
)

display(cl007_iqr_results)


,check,expected,observed,status
0,Reconstructed high-cost records,2610,2610,PASS
1,Existing high-cost flagged records,2610,2610,PASS
2,IQR flag mismatches,0,0,PASS


In [55]:

# 05 — Clarification Compliance
# CL-007: Outlier-independent financial eligibility

cl007_review_mask = (
    df["high_total_cost_outlier_flag"]
    | df["negative_gross_profit_flag"]
)

cl007_review_rows = df.loc[cl007_review_mask]

cl007_expected_certified = (
    cl007_review_rows["order_status"].eq("Completed")
    & ~cl007_review_rows["invalid_numeric_flag"]
)

cl007_expected_provisional = (
    cl007_review_rows["order_status"].isin(
        ["Completed", "Partially Returned"]
    )
    & ~cl007_review_rows["invalid_numeric_flag"]
)

cl007_eligibility_results = pd.DataFrame([
    {
        "check": "Certified eligibility mismatches",
        "expected": 0,
        "observed": int(
            (
                cl007_expected_certified
                != cl007_review_rows[
                    "certified_sales_eligible"
                ]
            ).sum()
        )
    },
    {
        "check": "Provisional eligibility mismatches",
        "expected": 0,
        "observed": int(
            (
                cl007_expected_provisional
                != cl007_review_rows[
                    "provisional_return_inclusive_eligible"
                ]
            ).sum()
        )
    }
])

cl007_eligibility_results["status"] = np.where(
    cl007_eligibility_results["expected"]
    == cl007_eligibility_results["observed"],
    "PASS",
    "FAIL"
)

display(cl007_eligibility_results)


,check,expected,observed,status
0,Certified eligibility mismatches,0,0,PASS
1,Provisional eligibility mismatches,0,0,PASS


In [56]:

# 05 — Clarification Compliance
# CL-007: Certified financial exposure assessment

cl007_certified_high_cost = df.loc[
    df["certified_sales_eligible"]
    & df["high_total_cost_outlier_flag"]
]

cl007_certified_negative_profit = df.loc[
    df["certified_sales_eligible"]
    & df["negative_gross_profit_flag"]
]

cl007_certified_combined = df.loc[
    df["certified_sales_eligible"]
    & df["high_cost_negative_profit_flag"]
]

cl007_exposure_populations = {
    "Certified high-cost outliers": cl007_certified_high_cost,
    "Certified negative gross profit": cl007_certified_negative_profit,
    "Certified combined anomalies": cl007_certified_combined
}

cl007_financial_exposure = pd.DataFrame([
    {
        "population": name,
        "records": len(data),
        "net_revenue": data["net_revenue"].sum(),
        "total_cost": data["total_cost"].sum(),
        "gross_profit": data["gross_profit"].sum()
    }
    for name, data in cl007_exposure_populations.items()
])

display(cl007_financial_exposure.round(2))


,population,records,net_revenue,total_cost,gross_profit
0,Certified high-cost outliers,2377,"5,562,170.35","3,899,828.54","1,662,341.81"
1,Certified negative gross profit,1553,"590,244.98","646,037.39","-55,792.41"
2,Certified combined anomalies,167,"234,929.66","255,870.96","-20,941.30"


In [57]:

# 05 — Clarification Compliance
# CL-007: Priority product investigation exposure

cl007_priority_products = [
    "P0151",
    "P0028",
    "P0083"
]

cl007_priority_rows = df.loc[
    df["product_id"].isin(cl007_priority_products)
    & cl007_review_mask
]

cl007_priority_summary = (
    cl007_priority_rows
    .groupby("product_id")
    .agg(
        flagged_records=("sales_line_id", "size"),
        high_cost_records=("high_total_cost_outlier_flag", "sum"),
        negative_profit_records=("negative_gross_profit_flag", "sum"),
        combined_anomaly_records=("high_cost_negative_profit_flag", "sum"),
        net_revenue=("net_revenue", "sum"),
        gross_profit=("gross_profit", "sum")
    )
    .reindex(cl007_priority_products, fill_value=0)
    .reset_index()
)

display(cl007_priority_summary.round(2))


,product_id,flagged_records,high_cost_records,negative_profit_records,combined_anomaly_records,net_revenue,gross_profit
0,P0151,70,35,55,20,"80,044.94",211.34
1,P0028,55,37,29,11,"78,001.43","3,394.83"
2,P0083,44,21,33,10,"43,684.40",714.62


In [58]:

# 05 — Clarification Compliance
# CL-007: Compliance validation summary

cl007_control_groups = {
    "Original anomaly exposure": cl007_exposure_results,
    "Negative profit flag accuracy": cl007_negative_flag_results,
    "Independent IQR flag accuracy": cl007_iqr_results,
    "Financial eligibility independence": cl007_eligibility_results
}

cl007_summary = pd.DataFrame([
    {
        "control": control,
        "status": (
            "PASS"
            if results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for control, results in cl007_control_groups.items()
])

display(cl007_summary)

print(
    "CL-007 analytical implementation:",
    "PASS"
    if cl007_summary["status"].eq("PASS").all()
    else "FAIL"
)


,control,status
0,Original anomaly exposure,PASS
1,Negative profit flag accuracy,PASS
2,Independent IQR flag accuracy,PASS
3,Financial eligibility independence,PASS


CL-007 analytical implementation: PASS



# 06 — Dimensional Integrity & Aggregation

**Objective:** Validate dimensional references, reporting attribution, and aggregation consistency to ensure that certified financial results remain accurate across business dimensions.

**Governing Reference:** IRDS-BR001 — Integrated Requirements & Data Specification; BR001 — Business Clarification Response & Approved Cleaning Decisions.

## Validation Scope

- Inspect available customer, product, channel, region, and date fields.
- Verify dimensional key completeness and reference coverage.
- Identify missing, unmatched, or inconsistent dimension references.
- Validate approved Unknown Customer and Unknown Channel attribution.
- Confirm that dimensional joins do not duplicate or unintentionally remove analytical records.
- Reconcile certified financial KPIs across reporting dimensions.
- Verify that dimensional aggregations preserve the certified financial baseline.

## Acceptance Principles

- Preserve the original analytical record population.
- Avoid unintended row multiplication or record loss during joins.
- Maintain approved Unknown attribution categories.
- Do not invent missing dimension identifiers.
- Reconcile dimensional totals against certified financial controls.
- Distinguish missing source references from reporting attribution.
- Escalate unresolved dimensional integrity issues before approving downstream analysis.


In [59]:

# 06 — Dimensional Integrity & Aggregation
# Dimensional schema and field availability inspection

dimension_keywords = [
    "customer",
    "product",
    "channel",
    "region",
    "date",
    "year",
    "month",
    "attribution"
]

dimensional_columns = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in dimension_keywords
    )
]

dimensional_schema = pd.DataFrame([
    {
        "column": col,
        "dtype": str(df[col].dtype),
        "null_count": int(df[col].isna().sum()),
        "distinct_values": int(df[col].nunique(dropna=True))
    }
    for col in dimensional_columns
])

display(dimensional_schema)

print("\nDimensional columns:", len(dimensional_columns))
print("Total dataset columns:", len(df.columns))


,column,dtype,null_count,distinct_values
0,order_date,str,0,362
1,customer_id,str,1,2500
2,product_id,str,0,180
3,channel_id,str,1,5
4,region_id,str,0,8
5,missing_customer_flag,bool,0,2
6,missing_channel_flag,bool,0,2
7,customer_attribution,str,0,2501
8,channel_attribution,str,0,6
9,order_customer_conflict_flag,bool,0,2



Dimensional columns: 11
Total dataset columns: 47


In [60]:

# 06 — Dimensional Integrity & Aggregation
# Dimensional reference completeness validation

dimension_key_fields = [
    "customer_id",
    "product_id",
    "channel_id",
    "region_id"
]

expected_missing_counts = {
    "customer_id": 1,
    "product_id": 0,
    "channel_id": 1,
    "region_id": 0
}

dimension_completeness_results = pd.DataFrame([
    {
        "dimension_key": field,
        "expected_missing": expected_missing_counts[field],
        "observed_missing": int(df[field].isna().sum())
    }
    for field in dimension_key_fields
])

dimension_completeness_results["status"] = np.where(
    dimension_completeness_results["expected_missing"]
    == dimension_completeness_results["observed_missing"],
    "PASS",
    "FAIL"
)

display(dimension_completeness_results)

print(
    "Dimensional reference completeness:",
    "PASS"
    if dimension_completeness_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension_key,expected_missing,observed_missing,status
0,customer_id,1,1,PASS
1,product_id,0,0,PASS
2,channel_id,1,1,PASS
3,region_id,0,0,PASS


Dimensional reference completeness: PASS


In [61]:

# 06 — Dimensional Integrity & Aggregation
# Dimensional reference source discovery

reference_keywords = [
    "dim_customer",
    "dim_product",
    "dim_channel",
    "dim_region",
    "dim_date"
]

reference_files = []

for path in PROJECT_ROOT.rglob("*"):
    if (
        path.is_file()
        and path.suffix.lower() in [".csv", ".parquet", ".db", ".sqlite"]
        and any(
            keyword in path.name.lower()
            for keyword in reference_keywords
        )
    ):
        reference_files.append({
            "filename": path.name,
            "relative_path": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "file_type": path.suffix.lower()
        })

reference_inventory = pd.DataFrame(
    reference_files,
    columns=["filename", "relative_path", "file_type"]
)

print("Dimension reference sources discovered:")
display(reference_inventory)

print("Total reference files:", len(reference_inventory))


Dimension reference sources discovered:


,filename,relative_path,file_type


Total reference files: 0


In [62]:

# 06 — Dimensional Integrity & Aggregation
# SQLite warehouse source discovery

warehouse_files = [
    path
    for path in PROJECT_ROOT.rglob("nova_analytics_warehouse.db")
    if path.is_file()
]

warehouse_inventory = pd.DataFrame([
    {
        "filename": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2)
    }
    for path in warehouse_files
])

print("SQLite warehouse files discovered:")
display(warehouse_inventory)

print("Total warehouse files:", len(warehouse_files))


SQLite warehouse files discovered:


,filename,relative_path,size_mb
0,nova_analytics_warehouse.db,04_data\nova_analytics_warehouse.db,5.38


Total warehouse files: 1


In [63]:

# 06 — Dimensional Integrity & Aggregation
# Warehouse dimension tables inspection

import sqlite3

warehouse_path = warehouse_files[0]

warehouse_connection = sqlite3.connect(
    f"file:{warehouse_path.as_posix()}?mode=ro",
    uri=True
)

expected_dimension_tables = [
    "dim_customer",
    "dim_product",
    "dim_channel",
    "dim_region",
    "dim_date"
]

warehouse_tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    warehouse_connection
)

dimension_table_results = pd.DataFrame([
    {
        "table": table,
        "exists": table in warehouse_tables["name"].values,
        "status": (
            "PASS"
            if table in warehouse_tables["name"].values
            else "FAIL"
        )
    }
    for table in expected_dimension_tables
])

display(dimension_table_results)

print(
    "Warehouse dimension tables:",
    "PASS"
    if dimension_table_results["status"].eq("PASS").all()
    else "FAIL"
)


,table,exists,status
0,dim_customer,True,PASS
1,dim_product,True,PASS
2,dim_channel,True,PASS
3,dim_region,True,PASS
4,dim_date,True,PASS


Warehouse dimension tables: PASS


In [64]:

# 06 — Dimensional Integrity & Aggregation
# Dimension key integrity and coverage validation

dimension_config = {
    "dim_customer": ("customer_id", "customer_id"),
    "dim_product": ("product_id", "product_id"),
    "dim_channel": ("channel_id", "channel_id"),
    "dim_region": ("region_id", "region_id"),
    "dim_date": ("full_date", "order_date")
}

dimension_key_results = []
dimension_reference_details = {}

for table, (dimension_key, analytical_key) in dimension_config.items():

    dimension_df = pd.read_sql_query(
        f'SELECT "{dimension_key}" FROM "{table}"',
        warehouse_connection
    )

    dimension_values = set(
        dimension_df[dimension_key].dropna()
    )

    analytical_values = set(
        df[analytical_key].dropna()
    )

    unmatched_values = analytical_values - dimension_values

    dimension_reference_details[table] = {
        "unmatched_values": sorted(unmatched_values),
        "dimension_records": len(dimension_df)
    }

    dimension_key_results.append({
        "dimension": table,
        "dimension_records": len(dimension_df),
        "null_keys": int(
            dimension_df[dimension_key].isna().sum()
        ),
        "duplicate_keys": int(
            dimension_df[dimension_key].duplicated().sum()
        ),
        "unmatched_analytical_keys": len(unmatched_values)
    })

dimension_key_results = pd.DataFrame(dimension_key_results)

dimension_key_results["status"] = np.where(
    (
        dimension_key_results["null_keys"].eq(0)
        & dimension_key_results["duplicate_keys"].eq(0)
        & dimension_key_results["unmatched_analytical_keys"].eq(0)
    ),
    "PASS",
    "FAIL"
)

display(dimension_key_results)

print(
    "Dimension key integrity:",
    "PASS"
    if dimension_key_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension,dimension_records,null_keys,duplicate_keys,unmatched_analytical_keys,status
0,dim_customer,2500,0,0,0,PASS
1,dim_product,180,0,0,0,PASS
2,dim_channel,5,0,0,0,PASS
3,dim_region,8,0,0,0,PASS
4,dim_date,546,0,0,0,PASS


Dimension key integrity: PASS


In [65]:

# 06 — Dimensional Integrity & Aggregation
# Dimensional join cardinality and record preservation

dimension_join_config = {
    "dim_customer": ("customer_id", "customer_id"),
    "dim_product": ("product_id", "product_id"),
    "dim_channel": ("channel_id", "channel_id"),
    "dim_region": ("region_id", "region_id"),
    "dim_date": ("full_date", "order_date")
}

dimension_join_results = []

for table, (dimension_key, analytical_key) in dimension_join_config.items():

    dimension_keys = pd.read_sql_query(
        f'SELECT "{dimension_key}" FROM "{table}"',
        warehouse_connection
    )

    dimension_keys["_dimension_match"] = True

    joined = df[
        ["analytical_row_key", analytical_key]
    ].merge(
        dimension_keys,
        how="left",
        left_on=analytical_key,
        right_on=dimension_key,
        validate="many_to_one"
    )

    original_rows = len(df)
    joined_rows = len(joined)

    unmatched_non_null = (
        joined[analytical_key].notna()
        & joined["_dimension_match"].isna()
    ).sum()

    dimension_join_results.append({
        "dimension": table,
        "original_rows": original_rows,
        "joined_rows": joined_rows,
        "row_difference": joined_rows - original_rows,
        "unmatched_non_null": int(unmatched_non_null),
        "distinct_analytical_keys": joined[
            "analytical_row_key"
        ].nunique()
    })

dimension_join_results = pd.DataFrame(dimension_join_results)

dimension_join_results["status"] = np.where(
    (
        dimension_join_results["row_difference"].eq(0)
        & dimension_join_results["unmatched_non_null"].eq(0)
        & dimension_join_results["distinct_analytical_keys"].eq(len(df))
    ),
    "PASS",
    "FAIL"
)

display(dimension_join_results)

print(
    "Dimensional join integrity:",
    "PASS"
    if dimension_join_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension,original_rows,joined_rows,row_difference,unmatched_non_null,distinct_analytical_keys,status
0,dim_customer,23530,23530,0,0,23530,PASS
1,dim_product,23530,23530,0,0,23530,PASS
2,dim_channel,23530,23530,0,0,23530,PASS
3,dim_region,23530,23530,0,0,23530,PASS
4,dim_date,23530,23530,0,0,23530,PASS


Dimensional join integrity: PASS


In [66]:

# 06 — Dimensional Integrity & Aggregation
# Unknown attribution and dimensional join exception validation

cl006_expected_exceptions = {
    "customer": {
        "sales_line_id": "SL0000105",
        "key": "customer_id",
        "flag": "missing_customer_flag",
        "attribution": "customer_attribution",
        "expected_label": "Unknown Customer"
    },
    "channel": {
        "sales_line_id": "SL0008422",
        "key": "channel_id",
        "flag": "missing_channel_flag",
        "attribution": "channel_attribution",
        "expected_label": "Unknown Channel"
    }
}

dimension_exception_results = []

for dimension, config in cl006_expected_exceptions.items():

    affected = df.loc[
        df["sales_line_id"].eq(config["sales_line_id"])
    ]

    record_found = len(affected) == 1

    key_missing = (
        record_found
        and affected[config["key"]].isna().all()
    )

    flag_correct = (
        record_found
        and affected[config["flag"]].eq(True).all()
    )

    attribution_correct = (
        record_found
        and affected[config["attribution"]]
        .eq(config["expected_label"]).all()
    )

    dimension_exception_results.append({
        "dimension": dimension,
        "record_found": record_found,
        "original_key_missing": key_missing,
        "missing_flag_correct": flag_correct,
        "attribution_correct": attribution_correct,
        "status": (
            "PASS"
            if all([
                record_found,
                key_missing,
                flag_correct,
                attribution_correct
            ])
            else "FAIL"
        )
    })

dimension_exception_results = pd.DataFrame(
    dimension_exception_results
)

display(dimension_exception_results)

print(
    "Dimensional exception handling:",
    "PASS"
    if dimension_exception_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension,record_found,original_key_missing,missing_flag_correct,attribution_correct,status
0,customer,True,True,True,True,PASS
1,channel,True,True,True,True,PASS


Dimensional exception handling: PASS


In [67]:

# 06 — Dimensional Integrity & Aggregation
# Dimensional financial aggregation reconciliation

cl068_financial_columns = [
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "total_cost",
    "gross_profit"
]

cl068_dimensions = [
    "customer_attribution",
    "product_id",
    "channel_attribution",
    "region_id",
    "order_date"
]

cl068_certified = df.loc[
    df["certified_sales_eligible"]
].copy()

cl068_baseline = (
    cl068_certified[cl068_financial_columns].sum()
)

cl068_results = []

for dimension in cl068_dimensions:

    aggregated = (
        cl068_certified
        .groupby(dimension, dropna=False)[cl068_financial_columns]
        .sum()
    )

    reconciled = aggregated.sum()

    for metric in cl068_financial_columns:

        difference = round(
            reconciled[metric] - cl068_baseline[metric],
            2
        )

        cl068_results.append({
            "dimension": dimension,
            "metric": metric,
            "baseline": round(cl068_baseline[metric], 2),
            "aggregated": round(reconciled[metric], 2),
            "difference": difference,
            "status": (
                "PASS"
                if abs(difference) < 0.005
                else "FAIL"
            )
        })

cl068_reconciliation_results = pd.DataFrame(
    cl068_results
)

display(cl068_reconciliation_results)

print(
    "Dimensional financial reconciliation:",
    "PASS"
    if cl068_reconciliation_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension,metric,baseline,aggregated,difference,status
0,customer_attribution,gross_revenue,"16,068,562.07","16,068,562.07",-0.00,PASS
1,customer_attribution,discount_amount,"2,504,281.22","2,504,281.22",0.00,PASS
2,customer_attribution,net_revenue,"13,564,280.85","13,564,280.85",0.00,PASS
3,customer_attribution,total_cost,"9,483,638.54","9,483,638.54",0.00,PASS
4,customer_attribution,gross_profit,"4,080,642.31","4,080,642.31",-0.00,PASS
5,product_id,gross_revenue,"16,068,562.07","16,068,562.07",0.00,PASS
6,product_id,discount_amount,"2,504,281.22","2,504,281.22",0.00,PASS
7,product_id,net_revenue,"13,564,280.85","13,564,280.85",0.00,PASS
8,product_id,total_cost,"9,483,638.54","9,483,638.54",-0.00,PASS
9,product_id,gross_profit,"4,080,642.31","4,080,642.31",0.00,PASS


Dimensional financial reconciliation: PASS


In [68]:

# 06 — Dimensional Integrity & Aggregation
# Dimensional reporting coverage and consistency validation

cl069_dimensions = [
    "customer_attribution",
    "product_id",
    "channel_attribution",
    "region_id",
    "order_date"
]

cl069_certified = df.loc[
    df["certified_sales_eligible"]
].copy()

cl069_baseline_count = len(cl069_certified)

cl069_results = []

for dimension in cl069_dimensions:

    grouped_counts = (
        cl069_certified
        .groupby(dimension, dropna=False)
        .size()
    )

    cl069_results.append({
        "dimension": dimension,
        "certified_records": cl069_baseline_count,
        "grouped_records": int(grouped_counts.sum()),
        "missing_reporting_values": int(
            cl069_certified[dimension].isna().sum()
        ),
        "distinct_categories": int(
            cl069_certified[dimension].nunique(dropna=False)
        )
    })

cl069_coverage_results = pd.DataFrame(cl069_results)

cl069_coverage_results["status"] = np.where(
    (
        cl069_coverage_results["grouped_records"]
        .eq(cl069_coverage_results["certified_records"])
        &
        cl069_coverage_results["missing_reporting_values"].eq(0)
    ),
    "PASS",
    "FAIL"
)

display(cl069_coverage_results)

print(
    "Dimensional reporting coverage:",
    "PASS"
    if cl069_coverage_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension,certified_records,grouped_records,missing_reporting_values,distinct_categories,status
0,customer_attribution,21342,21342,0,2501,PASS
1,product_id,21342,21342,0,180,PASS
2,channel_attribution,21342,21342,0,5,PASS
3,region_id,21342,21342,0,8,PASS
4,order_date,21342,21342,0,362,PASS


Dimensional reporting coverage: PASS


In [69]:

# 06 — Dimensional Integrity & Aggregation
# Dimensional validation summary

dimensional_control_groups = {
    "Reference completeness": dimension_completeness_results,
    "Warehouse dimension availability": dimension_table_results,
    "Dimension key integrity": dimension_key_results,
    "Join cardinality and preservation": dimension_join_results,
    "Unknown attribution exceptions": dimension_exception_results,
    "Financial aggregation reconciliation": cl068_reconciliation_results,
    "Reporting coverage and consistency": cl069_coverage_results
}

dimensional_validation_summary = pd.DataFrame([
    {
        "control": control,
        "checks": len(results),
        "passed": int(results["status"].eq("PASS").sum()),
        "failed": int(results["status"].eq("FAIL").sum()),
        "status": (
            "PASS"
            if len(results) > 0
            and results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for control, results in dimensional_control_groups.items()
])

display(dimensional_validation_summary)

print(
    "\nTotal checks:",
    int(dimensional_validation_summary["checks"].sum())
)

print(
    "Total passed:",
    int(dimensional_validation_summary["passed"].sum())
)

print(
    "Total failed:",
    int(dimensional_validation_summary["failed"].sum())
)

print(
    "\nSection 06 validation:",
    "PASS"
    if dimensional_validation_summary["status"].eq("PASS").all()
    else "FAIL"
)


,control,checks,passed,failed,status
0,Reference completeness,4,4,0,PASS
1,Warehouse dimension availability,5,5,0,PASS
2,Dimension key integrity,5,5,0,PASS
3,Join cardinality and preservation,5,5,0,PASS
4,Unknown attribution exceptions,2,2,0,PASS
5,Financial aggregation reconciliation,25,25,0,PASS
6,Reporting coverage and consistency,5,5,0,PASS



Total checks: 51
Total passed: 51
Total failed: 0

Section 06 validation: PASS


# 07 — KPI Readiness & Restrictions

**Objective:** Determine which business KPIs are approved for certified reporting, which require provisional treatment, and which remain restricted or blocked under the governing business clarifications.

**Governing Reference:** IRDS-BR001 — Integrated Requirements & Data Specification; BR001 — Business Clarification Response & Approved Cleaning Decisions.

## Validation Scope

- Verify certified and provisional KPI population boundaries.
- Confirm financial KPI readiness and reconciliation.
- Enforce order-based KPI restrictions under CL001.
- Preserve provisional financial treatment under CL002.
- Verify that approved data-quality exceptions do not introduce unauthorized exclusions.
- Establish explicit reporting permissions and limitations for downstream business analysis.

## Acceptance Principles

- Certified KPIs must use the approved certified population.
- Provisional financial results must be clearly distinguished from certified results.
- Order Count, Average Order Value, and order-frequency metrics remain blocked until order identity is resolved.
- No blocked KPI may be presented as certified.
- All reporting restrictions must remain traceable to approved business clarifications.
- KPI readiness must be established before business analysis begins.

In [70]:

# 07 — KPI Readiness & Restrictions
# Certified and provisional KPI population boundary validation

cl071_certified_expected = (
    df["order_status"].eq("Completed")
    & df["numeric_quality_eligible"].eq(True)
)

cl071_provisional_expected = (
    df["order_status"].isin([
        "Completed",
        "Partially Returned"
    ])
    & df["numeric_quality_eligible"].eq(True)
)

cl071_certified_actual = df[
    "certified_sales_eligible"
].eq(True)

cl071_provisional_actual = df[
    "provisional_return_inclusive_eligible"
].eq(True)

cl071_checks = [
    {
        "control": "Certified population",
        "expected_records": 21342,
        "actual_records": int(cl071_certified_actual.sum()),
        "rule_mismatches": int(
            (cl071_certified_actual != cl071_certified_expected).sum()
        )
    },
    {
        "control": "Provisional population",
        "expected_records": 22417,
        "actual_records": int(cl071_provisional_actual.sum()),
        "rule_mismatches": int(
            (cl071_provisional_actual != cl071_provisional_expected).sum()
        )
    },
    {
        "control": "Partially Returned population",
        "expected_records": 1075,
        "actual_records": int(
            df["order_status"].eq("Partially Returned").sum()
        ),
        "rule_mismatches": int(
            (
                df["order_status"].eq("Partially Returned")
                & ~cl071_provisional_actual
            ).sum()
        )
    }
]

cl071_results = pd.DataFrame(cl071_checks)

cl071_results["status"] = np.where(
    (
        cl071_results["expected_records"].eq(
            cl071_results["actual_records"]
        )
        & cl071_results["rule_mismatches"].eq(0)
    ),
    "PASS",
    "FAIL"
)

display(cl071_results)

print(
    "KPI population boundary:",
    "PASS"
    if cl071_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,expected_records,actual_records,rule_mismatches,status
0,Certified population,21342,21342,0,PASS
1,Provisional population,22417,22417,0,PASS
2,Partially Returned population,1075,1075,0,PASS


KPI population boundary: PASS


In [71]:

# 07 — KPI Readiness & Restrictions
# Order-based KPI restriction validation (CL001)

cl072_required_columns = [
    "order_id",
    "order_key_verified",
    "order_kpi_eligible"
]

cl072_missing_columns = [
    column for column in cl072_required_columns
    if column not in df.columns
]

if cl072_missing_columns:
    raise KeyError(
        f"Missing required columns: {cl072_missing_columns}"
    )

cl072_results = pd.DataFrame([
    {
        "control": "Unverified order keys",
        "expected": len(df),
        "actual": int((~df["order_key_verified"]).sum())
    },
    {
        "control": "Order KPI eligible records",
        "expected": 0,
        "actual": int(df["order_kpi_eligible"].sum())
    },
    {
        "control": "Certified records with order KPI permission",
        "expected": 0,
        "actual": int(
            (
                df["certified_sales_eligible"]
                & df["order_kpi_eligible"]
            ).sum()
        )
    },
    {
        "control": "Provisional records with order KPI permission",
        "expected": 0,
        "actual": int(
            (
                df["provisional_return_inclusive_eligible"]
                & df["order_kpi_eligible"]
            ).sum()
        )
    }
])

cl072_results["status"] = np.where(
    cl072_results["expected"].eq(cl072_results["actual"]),
    "PASS",
    "FAIL"
)

display(cl072_results)

print(
    "Order-based KPI restrictions:",
    "PASS"
    if cl072_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,expected,actual,status
0,Unverified order keys,23530,23530,PASS
1,Order KPI eligible records,0,0,PASS
2,Certified records with order KPI permission,0,0,PASS
3,Provisional records with order KPI permission,0,0,PASS


Order-based KPI restrictions: PASS


In [72]:

# 07 — KPI Readiness & Restrictions
# Provisional financial KPI restrictions validation (CL002)

cl073_partial_mask = df["order_status"].eq("Partially Returned")
cl073_certified_mask = df["certified_sales_eligible"].eq(True)
cl073_provisional_mask = df[
    "provisional_return_inclusive_eligible"
].eq(True)

cl073_partial = df.loc[cl073_partial_mask]

cl073_financial_columns = [
    "net_revenue",
    "total_cost",
    "gross_profit"
]

cl073_expected_amounts = {
    "net_revenue": 673506.50,
    "total_cost": 470229.86,
    "gross_profit": 203276.64
}

cl073_results = []

cl073_population_checks = {
    "Partial return records": (
        len(cl073_partial), 1075
    ),
    "Partial returns included in provisional": (
        int((cl073_partial_mask & cl073_provisional_mask).sum()),
        1075
    ),
    "Partial returns included in certified": (
        int((cl073_partial_mask & cl073_certified_mask).sum()),
        0
    ),
    "Certified population subset of provisional": (
        int((cl073_certified_mask & ~cl073_provisional_mask).sum()),
        0
    )
}

for control, (actual, expected) in cl073_population_checks.items():
    cl073_results.append({
        "control": control,
        "expected": expected,
        "actual": actual,
        "difference": actual - expected,
        "status": "PASS" if actual == expected else "FAIL"
    })

for metric in cl073_financial_columns:
    actual = round(float(cl073_partial[metric].sum()), 2)
    expected = cl073_expected_amounts[metric]

    cl073_results.append({
        "control": f"Partial returns {metric}",
        "expected": expected,
        "actual": actual,
        "difference": round(actual - expected, 2),
        "status": (
            "PASS"
            if abs(actual - expected) < 0.005
            else "FAIL"
        )
    })

cl073_results = pd.DataFrame(cl073_results)

display(cl073_results)

print(
    "Provisional financial KPI restrictions:",
    "PASS"
    if cl073_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,expected,actual,difference,status
0,Partial return records,"1,075.00","1,075.00",0.00,PASS
1,Partial returns included in provisional,"1,075.00","1,075.00",0.00,PASS
2,Partial returns included in certified,0.00,0.00,0.00,PASS
3,Certified population subset of provisional,0.00,0.00,0.00,PASS
4,Partial returns net_revenue,"673,506.50","673,506.50",0.00,PASS
5,Partial returns total_cost,"470,229.86","470,229.86",0.00,PASS
6,Partial returns gross_profit,"203,276.64","203,276.64",0.00,PASS


Provisional financial KPI restrictions: PASS


In [73]:

# 07 — KPI Readiness & Restrictions
# Certified financial KPI readiness validation

cl074_certified = df.loc[
    df["certified_sales_eligible"].eq(True)
].copy()

cl074_expected = {
    "gross_revenue": 16068562.07,
    "discount_amount": 2504281.22,
    "net_revenue": 13564280.85,
    "total_cost": 9483638.54,
    "gross_profit": 4080642.31
}

cl074_actual = {
    metric: round(float(cl074_certified[metric].sum()), 2)
    for metric in cl074_expected
}

cl074_results = []

for metric, expected in cl074_expected.items():

    actual = cl074_actual[metric]
    difference = round(actual - expected, 2)

    cl074_results.append({
        "control": metric,
        "expected": expected,
        "actual": actual,
        "difference": difference,
        "status": "PASS" if abs(difference) < 0.005 else "FAIL"
    })

cl074_relationships = {
    "Net revenue = gross revenue - discounts": (
        cl074_actual["net_revenue"],
        cl074_actual["gross_revenue"]
        - cl074_actual["discount_amount"]
    ),
    "Gross profit = net revenue - total cost": (
        cl074_actual["gross_profit"],
        cl074_actual["net_revenue"]
        - cl074_actual["total_cost"]
    )
}

for control, (actual, expected) in cl074_relationships.items():

    difference = round(actual - expected, 2)

    cl074_results.append({
        "control": control,
        "expected": round(expected, 2),
        "actual": round(actual, 2),
        "difference": difference,
        "status": "PASS" if abs(difference) < 0.005 else "FAIL"
    })

cl074_results = pd.DataFrame(cl074_results)

display(cl074_results)

print(
    "Certified financial KPI readiness:",
    "PASS"
    if cl074_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,expected,actual,difference,status
0,gross_revenue,"16,068,562.07","16,068,562.07",0.00,PASS
1,discount_amount,"2,504,281.22","2,504,281.22",0.00,PASS
2,net_revenue,"13,564,280.85","13,564,280.85",0.00,PASS
3,total_cost,"9,483,638.54","9,483,638.54",0.00,PASS
4,gross_profit,"4,080,642.31","4,080,642.31",0.00,PASS
5,Net revenue = gross revenue - discounts,"13,564,280.85","13,564,280.85",0.00,PASS
6,Gross profit = net revenue - total cost,"4,080,642.31","4,080,642.31",-0.00,PASS


Certified financial KPI readiness: PASS


In [74]:

# 07 — KPI Readiness & Restrictions
# KPI reporting permissions and restrictions validation

cl075_dependencies = {
    "Certified population": cl071_results["status"].eq("PASS").all(),
    "Order restrictions": cl072_results["status"].eq("PASS").all(),
    "Provisional treatment": cl073_results["status"].eq("PASS").all(),
    "Certified financial reconciliation": cl074_results["status"].eq("PASS").all()
}

cl075_permissions = [
    {
        "kpi": "Gross Revenue",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / CL002 / CL005"
    },
    {
        "kpi": "Discount Amount",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / CL002 / CL005"
    },
    {
        "kpi": "Net Revenue",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / CL002 / CL005"
    },
    {
        "kpi": "Total Cost",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / CL002 / CL005"
    },
    {
        "kpi": "Gross Profit",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / CL002 / CL005"
    },
    {
        "kpi": "Return-Inclusive Financial Results",
        "reporting_status": "PROVISIONAL",
        "population": "Provisional",
        "governing_rule": "CL002"
    },
    {
        "kpi": "Order Count",
        "reporting_status": "BLOCKED",
        "population": "Not authorized",
        "governing_rule": "CL001"
    },
    {
        "kpi": "Average Order Value",
        "reporting_status": "BLOCKED",
        "population": "Not authorized",
        "governing_rule": "CL001"
    },
    {
        "kpi": "Order Frequency",
        "reporting_status": "BLOCKED",
        "population": "Not authorized",
        "governing_rule": "CL001"
    }
]

cl075_results = pd.DataFrame(cl075_permissions)

cl075_valid_statuses = {
    "CERTIFIED",
    "PROVISIONAL",
    "BLOCKED"
}

cl075_checks = {
    "Required validations passed": all(cl075_dependencies.values()),
    "Valid reporting classifications": (
        cl075_results["reporting_status"]
        .isin(cl075_valid_statuses).all()
    ),
    "Five certified financial KPIs": (
        cl075_results["reporting_status"].eq("CERTIFIED").sum() == 5
    ),
    "Provisional financial treatment retained": (
        cl075_results["reporting_status"].eq("PROVISIONAL").sum() == 1
    ),
    "Three order-based KPIs blocked": (
        cl075_results["reporting_status"].eq("BLOCKED").sum() == 3
    )
}

cl075_validation = pd.DataFrame([
    {
        "control": control,
        "status": "PASS" if passed else "FAIL"
    }
    for control, passed in cl075_checks.items()
])

display(cl075_results)
display(cl075_validation)

print(
    "KPI reporting permissions:",
    "PASS"
    if cl075_validation["status"].eq("PASS").all()
    else "FAIL"
)


,kpi,reporting_status,population,governing_rule
0,Gross Revenue,CERTIFIED,Certified,IRDS / CL002 / CL005
1,Discount Amount,CERTIFIED,Certified,IRDS / CL002 / CL005
2,Net Revenue,CERTIFIED,Certified,IRDS / CL002 / CL005
3,Total Cost,CERTIFIED,Certified,IRDS / CL002 / CL005
4,Gross Profit,CERTIFIED,Certified,IRDS / CL002 / CL005
5,Return-Inclusive Financial Results,PROVISIONAL,Provisional,CL002
6,Order Count,BLOCKED,Not authorized,CL001
7,Average Order Value,BLOCKED,Not authorized,CL001
8,Order Frequency,BLOCKED,Not authorized,CL001


,control,status
0,Required validations passed,PASS
1,Valid reporting classifications,PASS
2,Five certified financial KPIs,PASS
3,Provisional financial treatment retained,PASS
4,Three order-based KPIs blocked,PASS


KPI reporting permissions: PASS


In [75]:

# 07 — KPI Readiness & Restrictions
# KPI eligibility and restriction consistency validation

cl076_certified = df["certified_sales_eligible"].eq(True)
cl076_provisional = df[
    "provisional_return_inclusive_eligible"
].eq(True)
cl076_numeric_valid = df["numeric_quality_eligible"].eq(True)
cl076_invalid_numeric = df["invalid_numeric_flag"].eq(True)
cl076_order_eligible = df["order_kpi_eligible"].eq(True)
cl076_order_verified = df["order_key_verified"].eq(True)

cl076_checks = {
    "Certified records with invalid numeric values": (
        cl076_certified & cl076_invalid_numeric
    ),
    "Provisional records with invalid numeric values": (
        cl076_provisional & cl076_invalid_numeric
    ),
    "Certified records failing numeric eligibility": (
        cl076_certified & ~cl076_numeric_valid
    ),
    "Provisional records failing numeric eligibility": (
        cl076_provisional & ~cl076_numeric_valid
    ),
    "Certified records outside Completed status": (
        cl076_certified & ~df["order_status"].eq("Completed")
    ),
    "Provisional records outside approved statuses": (
        cl076_provisional
        & ~df["order_status"].isin([
            "Completed",
            "Partially Returned"
        ])
    ),
    "Certified records excluded from provisional": (
        cl076_certified & ~cl076_provisional
    ),
    "Order KPI eligibility without verified order key": (
        cl076_order_eligible & ~cl076_order_verified
    ),
    "Order KPI eligibility despite CL001 restriction": (
        cl076_order_eligible
    ),
    "Numeric quality flag contradictions": (
        cl076_numeric_valid == cl076_invalid_numeric
    )
}

cl076_results = pd.DataFrame([
    {
        "control": control,
        "violations": int(mask.sum()),
        "expected_violations": 0
    }
    for control, mask in cl076_checks.items()
])

cl076_results["status"] = np.where(
    cl076_results["violations"].eq(
        cl076_results["expected_violations"]
    ),
    "PASS",
    "FAIL"
)

display(cl076_results)

print(
    "KPI eligibility consistency:",
    "PASS"
    if cl076_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,violations,expected_violations,status
0,Certified records with invalid numeric values,0,0,PASS
1,Provisional records with invalid numeric values,0,0,PASS
2,Certified records failing numeric eligibility,0,0,PASS
3,Provisional records failing numeric eligibility,0,0,PASS
4,Certified records outside Completed status,0,0,PASS
5,Provisional records outside approved statuses,0,0,PASS
6,Certified records excluded from provisional,0,0,PASS
7,Order KPI eligibility without verified order key,0,0,PASS
8,Order KPI eligibility despite CL001 restriction,0,0,PASS
9,Numeric quality flag contradictions,0,0,PASS


KPI eligibility consistency: PASS


In [76]:

# 07 — KPI Readiness & Restrictions
# KPI readiness and restrictions validation summary

cl077_control_groups = {
    "KPI population boundaries": cl071_results,
    "Order-based KPI restrictions": cl072_results,
    "Provisional financial restrictions": cl073_results,
    "Certified financial KPI readiness": cl074_results,
    "KPI reporting permissions": cl075_validation,
    "KPI eligibility consistency": cl076_results
}

cl077_summary = pd.DataFrame([
    {
        "control": name,
        "checks": len(results),
        "passed": int(results["status"].eq("PASS").sum()),
        "failed": int(results["status"].eq("FAIL").sum()),
        "status": (
            "PASS"
            if len(results) > 0
            and results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for name, results in cl077_control_groups.items()
])

display(cl077_summary)

cl077_total_checks = int(cl077_summary["checks"].sum())
cl077_total_passed = int(cl077_summary["passed"].sum())
cl077_total_failed = int(cl077_summary["failed"].sum())

cl077_section_status = (
    "PASS"
    if cl077_total_checks > 0
    and cl077_total_failed == 0
    and cl077_total_passed == cl077_total_checks
    else "FAIL"
)

print(f"\nTotal checks: {cl077_total_checks}")
print(f"Total passed: {cl077_total_passed}")
print(f"Total failed: {cl077_total_failed}")
print(f"\nSection 07 validation: {cl077_section_status}")


,control,checks,passed,failed,status
0,KPI population boundaries,3,3,0,PASS
1,Order-based KPI restrictions,4,4,0,PASS
2,Provisional financial restrictions,7,7,0,PASS
3,Certified financial KPI readiness,7,7,0,PASS
4,KPI reporting permissions,5,5,0,PASS
5,KPI eligibility consistency,10,10,0,PASS



Total checks: 36
Total passed: 36
Total failed: 0

Section 07 validation: PASS


# 08 — Validation Summary & Execution Gate

**Objective:** Consolidate the completed analytical validation results and determine whether BR001 is authorized to proceed to business analysis under the approved requirements, clarifications, and reporting restrictions.

**Governing Reference:** IRDS-BR001 — Integrated Requirements & Data Specification; BR001 — Business Clarification Response & Approved Cleaning Decisions; BR001 Analytical Plan.

## Validation Scope

- Confirm completion and successful execution of analytical validation sections.
- Verify certified financial KPI readiness.
- Confirm dimensional integrity and reporting coverage.
- Preserve approved clarification treatments and unresolved source-level issues.
- Review KPI permissions against the business questions and reporting requirements.
- Identify outstanding restrictions and dependencies.
- Establish the final execution decision and permitted analytical scope.

## Execution Gate Principles

- **GO:** All required controls pass and no material reporting restrictions remain.
- **CONDITIONAL GO:** Required controls pass, but analysis must proceed within explicitly documented restrictions.
- **NO-GO:** Critical validation failures or unresolved dependencies prevent reliable analysis.

## Required Gate Evidence

- Completed analytical validation results.
- Certified and provisional population boundaries.
- Financial KPI reconciliation.
- Dimensional validation summary.
- KPI reporting permissions.
- Outstanding clarification restrictions.
- Business-question and KPI scope coverage.

**Acceptance Principle:** Technical validation success does not automatically imply unrestricted business reporting authorization.


In [77]:

# 08 — Validation Summary & Execution Gate
# Dimensional and KPI validation evidence consolidation

cl078_sections = {
    "06 — Dimensional Integrity & Aggregation": dimensional_validation_summary,
    "07 — KPI Readiness & Restrictions": cl077_summary
}

cl078_expected_checks = {
    "06 — Dimensional Integrity & Aggregation": 51,
    "07 — KPI Readiness & Restrictions": 36
}

cl078_results = []

for section, results in cl078_sections.items():

    total_checks = int(results["checks"].sum())
    total_passed = int(results["passed"].sum())
    total_failed = int(results["failed"].sum())

    expected_checks = cl078_expected_checks[section]

    status = (
        "PASS"
        if total_checks == expected_checks
        and total_passed == total_checks
        and total_failed == 0
        and results["status"].eq("PASS").all()
        else "FAIL"
    )

    cl078_results.append({
        "section": section,
        "expected_checks": expected_checks,
        "actual_checks": total_checks,
        "passed": total_passed,
        "failed": total_failed,
        "status": status
    })

cl078_results = pd.DataFrame(cl078_results)

display(cl078_results)

print(
    "\nConsolidated checks:",
    int(cl078_results["actual_checks"].sum())
)

print(
    "Consolidated passed:",
    int(cl078_results["passed"].sum())
)

print(
    "Consolidated failed:",
    int(cl078_results["failed"].sum())
)

print(
    "\nValidation evidence consolidation:",
    "PASS"
    if cl078_results["status"].eq("PASS").all()
    else "FAIL"
)


,section,expected_checks,actual_checks,passed,failed,status
0,06 — Dimensional Integrity & Aggregation,51,51,51,0,PASS
1,07 — KPI Readiness & Restrictions,36,36,36,0,PASS



Consolidated checks: 87
Consolidated passed: 87
Consolidated failed: 0

Validation evidence consolidation: PASS


In [78]:

# 08 — Validation Summary & Execution Gate
# Required KPI source field inspection

cl079_required_fields = [
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "total_cost",
    "gross_profit",
    "quantity",
    "certified_sales_eligible"
]

cl079_field_results = pd.DataFrame([
    {
        "field": field,
        "exists": field in df.columns,
        "dtype": (
            str(df[field].dtype)
            if field in df.columns
            else "MISSING"
        ),
        "null_count": (
            int(df[field].isna().sum())
            if field in df.columns
            else None
        )
    }
    for field in cl079_required_fields
])

display(cl079_field_results)

print(
    "Required KPI fields available:",
    "PASS"
    if cl079_field_results["exists"].all()
    else "FAIL"
)


,field,exists,dtype,null_count
0,gross_revenue,True,float64,0
1,discount_amount,True,float64,0
2,net_revenue,True,float64,0
3,total_cost,True,float64,0
4,gross_profit,True,float64,0
5,quantity,True,int64,0
6,certified_sales_eligible,True,bool,0


Required KPI fields available: PASS


In [79]:

# 08 — Validation Summary & Execution Gate
# Derived KPI calculation and units sold validation

cl079b_certified = df.loc[
    df["certified_sales_eligible"].eq(True)
].copy()

cl079b_totals = cl079b_certified[
    [
        "gross_revenue",
        "discount_amount",
        "net_revenue",
        "gross_profit",
        "quantity"
    ]
].sum()

cl079b_gross_margin = (
    cl079b_totals["gross_profit"]
    / cl079b_totals["net_revenue"] * 100
    if cl079b_totals["net_revenue"] != 0
    else np.nan
)

cl079b_discount_rate = (
    cl079b_totals["discount_amount"]
    / cl079b_totals["gross_revenue"] * 100
    if cl079b_totals["gross_revenue"] != 0
    else np.nan
)

cl079b_units_sold = int(cl079b_totals["quantity"])

cl079b_expected_gross_margin = (
    4080642.31 / 13564280.85 * 100
)

cl079b_expected_discount_rate = (
    2504281.22 / 16068562.07 * 100
)

cl079b_checks = [
    {
        "control": "Gross Margin %",
        "actual": cl079b_gross_margin,
        "expected": cl079b_expected_gross_margin,
        "tolerance": 0.000001
    },
    {
        "control": "Discount Rate %",
        "actual": cl079b_discount_rate,
        "expected": cl079b_expected_discount_rate,
        "tolerance": 0.000001
    },
    {
        "control": "Units Sold aggregation",
        "actual": cl079b_units_sold,
        "expected": int(
            cl079b_certified["quantity"].sum()
        ),
        "tolerance": 0
    },
    {
        "control": "Certified quantity eligibility",
        "actual": int(
            (cl079b_certified["quantity"] <= 0).sum()
        ),
        "expected": 0,
        "tolerance": 0
    }
]

cl079b_results = pd.DataFrame(cl079b_checks)

cl079b_results["difference"] = (
    cl079b_results["actual"]
    - cl079b_results["expected"]
)

cl079b_results["status"] = np.where(
    cl079b_results["difference"].abs()
    <= cl079b_results["tolerance"],
    "PASS",
    "FAIL"
)

display(cl079b_results)

print(f"\nGross Margin: {cl079b_gross_margin:.4f}%")
print(f"Discount Rate: {cl079b_discount_rate:.4f}%")
print(f"Units Sold: {cl079b_units_sold:,}")

print(
    "\nDerived KPI validation:",
    "PASS"
    if cl079b_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,actual,expected,tolerance,difference,status
0,Gross Margin %,30.08,30.08,0.00,0.00,PASS
1,Discount Rate %,15.58,15.58,0.00,-0.00,PASS
2,Units Sold aggregation,"45,082.00","45,082.00",0.00,0.00,PASS
3,Certified quantity eligibility,0.00,0.00,0.00,0.00,PASS



Gross Margin: 30.0837%
Discount Rate: 15.5850%
Units Sold: 45,082

Derived KPI validation: PASS


In [80]:

# 08 — Validation Summary & Execution Gate
# Units sold dimensional and reporting period reconciliation

cl079c_certified = df.loc[
    df["certified_sales_eligible"].eq(True)
].copy()

cl079c_certified["_reporting_date"] = pd.to_datetime(
    cl079c_certified["order_date"],
    errors="coerce"
)

cl079c_certified["_reporting_year"] = (
    cl079c_certified["_reporting_date"].dt.year
)

cl079c_certified["_reporting_month"] = (
    cl079c_certified["_reporting_date"].dt.to_period("M").astype(str)
)

cl079c_baseline = 45082

cl079c_dimensions = [
    "_reporting_year",
    "_reporting_month",
    "product_id",
    "channel_attribution",
    "customer_attribution",
    "region_id"
]

cl079c_results = []

for dimension in cl079c_dimensions:

    grouped = (
        cl079c_certified
        .groupby(dimension, dropna=False)["quantity"]
        .sum()
    )

    actual = int(grouped.sum())

    cl079c_results.append({
        "control": f"Units Sold by {dimension}",
        "expected": cl079c_baseline,
        "actual": actual,
        "difference": actual - cl079c_baseline,
        "status": (
            "PASS" if actual == cl079c_baseline else "FAIL"
        )
    })

cl079c_valid_dates = (
    cl079c_certified["_reporting_date"].notna()
    & cl079c_certified["_reporting_year"].isin([2025, 2026])
    & cl079c_certified["_reporting_date"].dt.month.between(1, 6)
)

cl079c_invalid_dates = int((~cl079c_valid_dates).sum())

cl079c_results.append({
    "control": "Records outside approved H1 periods",
    "expected": 0,
    "actual": cl079c_invalid_dates,
    "difference": cl079c_invalid_dates,
    "status": (
        "PASS" if cl079c_invalid_dates == 0 else "FAIL"
    )
})

cl079c_results = pd.DataFrame(cl079c_results)

display(cl079c_results)

print("\nCertified units by year:")
display(
    cl079c_certified
    .groupby("_reporting_year")["quantity"]
    .sum()
    .rename("units_sold")
    .reset_index()
)

print(
    "\nUnits Sold dimensional reconciliation:",
    "PASS"
    if cl079c_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,expected,actual,difference,status
0,Units Sold by _reporting_year,45082,45082,0,PASS
1,Units Sold by _reporting_month,45082,45082,0,PASS
2,Units Sold by product_id,45082,45082,0,PASS
3,Units Sold by channel_attribution,45082,45082,0,PASS
4,Units Sold by customer_attribution,45082,45082,0,PASS
5,Units Sold by region_id,45082,45082,0,PASS
6,Records outside approved H1 periods,0,0,0,PASS



Certified units by year:


,_reporting_year,units_sold
0,2025,21695
1,2026,23387



Units Sold dimensional reconciliation: PASS


In [81]:

# 08 — Validation Summary & Execution Gate
# Extended KPI reporting permissions validation

cl079d_additional_kpis = pd.DataFrame([
    {
        "kpi": "Gross Margin %",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / KPI Contract"
    },
    {
        "kpi": "Discount Rate",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / KPI Contract"
    },
    {
        "kpi": "Units Sold",
        "reporting_status": "CERTIFIED",
        "population": "Certified",
        "governing_rule": "IRDS / KPI Contract"
    }
])

cl079d_permissions = pd.concat(
    [cl075_results, cl079d_additional_kpis],
    ignore_index=True
)

cl079d_dependencies = {
    "Original permissions validated": (
        cl075_validation["status"].eq("PASS").all()
    ),
    "Required KPI fields validated": (
        cl079_field_results["exists"].all()
    ),
    "Derived KPI calculations validated": (
        cl079b_results["status"].eq("PASS").all()
    ),
    "Units Sold reconciliation validated": (
        cl079c_results["status"].eq("PASS").all()
    ),
    "Eight certified KPIs": (
        cl079d_permissions["reporting_status"]
        .eq("CERTIFIED").sum() == 8
    ),
    "Provisional treatment preserved": (
        cl079d_permissions["reporting_status"]
        .eq("PROVISIONAL").sum() == 1
    ),
    "Three restricted KPIs preserved": (
        cl079d_permissions["reporting_status"]
        .eq("BLOCKED").sum() == 3
    ),
    "No duplicate KPI definitions": (
        cl079d_permissions["kpi"].is_unique
    )
}

cl079d_validation = pd.DataFrame([
    {
        "control": control,
        "status": "PASS" if result else "FAIL"
    }
    for control, result in cl079d_dependencies.items()
])

display(cl079d_permissions)
display(cl079d_validation)

print(
    "Extended KPI reporting permissions:",
    "PASS"
    if cl079d_validation["status"].eq("PASS").all()
    else "FAIL"
)


,kpi,reporting_status,population,governing_rule
0,Gross Revenue,CERTIFIED,Certified,IRDS / CL002 / CL005
1,Discount Amount,CERTIFIED,Certified,IRDS / CL002 / CL005
2,Net Revenue,CERTIFIED,Certified,IRDS / CL002 / CL005
3,Total Cost,CERTIFIED,Certified,IRDS / CL002 / CL005
4,Gross Profit,CERTIFIED,Certified,IRDS / CL002 / CL005
5,Return-Inclusive Financial Results,PROVISIONAL,Provisional,CL002
6,Order Count,BLOCKED,Not authorized,CL001
7,Average Order Value,BLOCKED,Not authorized,CL001
8,Order Frequency,BLOCKED,Not authorized,CL001
9,Gross Margin %,CERTIFIED,Certified,IRDS / KPI Contract


,control,status
0,Original permissions validated,PASS
1,Required KPI fields validated,PASS
2,Derived KPI calculations validated,PASS
3,Units Sold reconciliation validated,PASS
4,Eight certified KPIs,PASS
5,Provisional treatment preserved,PASS
6,Three restricted KPIs preserved,PASS
7,No duplicate KPI definitions,PASS


Extended KPI reporting permissions: PASS


In [82]:

# 08 — Validation Summary & Execution Gate
# Sales target warehouse structure inspection

cl080_table = "fact_sales_targets"

cl080_schema = pd.read_sql_query(
    f'PRAGMA table_info("{cl080_table}")',
    warehouse_connection
)

cl080_sample = pd.read_sql_query(
    f'SELECT * FROM "{cl080_table}" LIMIT 5',
    warehouse_connection
)

cl080_record_count = pd.read_sql_query(
    f'SELECT COUNT(*) AS record_count FROM "{cl080_table}"',
    warehouse_connection
)

print("Sales target table schema:")
display(
    cl080_schema[
        ["name", "type", "notnull", "pk"]
    ]
)

print("\nSales target sample:")
display(cl080_sample)

print("\nSales target record count:")
display(cl080_record_count)

print(
    "\nSales target structure inspection:",
    "PASS"
    if len(cl080_schema) > 0
    and int(cl080_record_count["record_count"].iloc[0]) > 0
    else "FAIL"
)


Sales target table schema:


,name,type,notnull,pk
0,target_id,TEXT,0,0
1,target_month,TEXT,0,0
2,channel_id,TEXT,0,0
3,country,TEXT,0,0
4,net_revenue_target,REAL,0,0
5,gross_profit_target,REAL,0,0



Sales target sample:


,target_id,target_month,channel_id,country,net_revenue_target,gross_profit_target
0,T20250101,2025-01,CH01,All Markets,"416,912.09","105,083.28"
1,T20250102,2025-01,CH02,All Markets,"457,590.20","115,940.67"
2,T20250103,2025-01,CH03,All Markets,"314,321.33","81,286.30"
3,T20250104,2025-01,CH04,All Markets,"245,646.93","64,164.64"
4,T20250105,2025-01,CH05,All Markets,"106,734.81","27,461.09"



Sales target record count:


,record_count
0,60



Sales target structure inspection: PASS


In [83]:

# 08 — Validation Summary & Execution Gate
# Sales target grain and integrity validation

cl081_targets = pd.read_sql_query(
    "SELECT * FROM fact_sales_targets",
    warehouse_connection
)

cl081_expected_months = {
    f"{year}-{month:02d}"
    for year in [2025, 2026]
    for month in range(1, 7)
}

cl081_channels = set(
    pd.read_sql_query(
        "SELECT channel_id FROM dim_channel",
        warehouse_connection
    )["channel_id"].dropna()
)

cl081_actual_pairs = set(
    zip(
        cl081_targets["target_month"],
        cl081_targets["channel_id"]
    )
)

cl081_expected_pairs = {
    (month, channel)
    for month in cl081_expected_months
    for channel in cl081_channels
}

cl081_checks = {
    "Expected target records": (
        len(cl081_targets), 60
    ),
    "Missing target identifiers": (
        int(cl081_targets["target_id"].isna().sum()), 0
    ),
    "Duplicate target identifiers": (
        int(cl081_targets["target_id"].duplicated().sum()), 0
    ),
    "Missing month-channel keys": (
        int(
            cl081_targets[
                ["target_month", "channel_id"]
            ].isna().any(axis=1).sum()
        ), 0
    ),
    "Duplicate month-channel combinations": (
        int(
            cl081_targets.duplicated(
                ["target_month", "channel_id"]
            ).sum()
        ), 0
    ),
    "Months outside approved periods": (
        int(
            (~cl081_targets["target_month"].isin(
                cl081_expected_months
            )).sum()
        ), 0
    ),
    "Channels outside approved dimension": (
        int(
            (
                ~cl081_targets["channel_id"].isin(
                    cl081_channels
                )
            ).sum()
        ), 0
    ),
    "Non-All Markets records": (
        int(
            cl081_targets["country"].ne("All Markets").sum()
        ), 0
    ),
    "Missing month-channel combinations": (
        len(cl081_expected_pairs - cl081_actual_pairs), 0
    ),
    "Unexpected month-channel combinations": (
        len(cl081_actual_pairs - cl081_expected_pairs), 0
    ),
    "Invalid net revenue targets": (
        int(
            (
                cl081_targets["net_revenue_target"].isna()
                | (cl081_targets["net_revenue_target"] < 0)
            ).sum()
        ), 0
    ),
    "Invalid gross profit targets": (
        int(
            (
                cl081_targets["gross_profit_target"].isna()
                | (cl081_targets["gross_profit_target"] < 0)
            ).sum()
        ), 0
    )
}

cl081_results = pd.DataFrame([
    {
        "control": control,
        "actual": actual,
        "expected": expected,
        "status": (
            "PASS" if actual == expected else "FAIL"
        )
    }
    for control, (actual, expected) in cl081_checks.items()
])

display(cl081_results)

print(
    "\nSales target grain and integrity:",
    "PASS"
    if cl081_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,actual,expected,status
0,Expected target records,60,60,PASS
1,Missing target identifiers,0,0,PASS
2,Duplicate target identifiers,0,0,PASS
3,Missing month-channel keys,0,0,PASS
4,Duplicate month-channel combinations,0,0,PASS
5,Months outside approved periods,0,0,PASS
6,Channels outside approved dimension,0,0,PASS
7,Non-All Markets records,0,0,PASS
8,Missing month-channel combinations,0,0,PASS
9,Unexpected month-channel combinations,0,0,PASS



Sales target grain and integrity: PASS


In [84]:

# 08 — Validation Summary & Execution Gate
# Sales actuals and targets grain reconciliation

if not cl081_results["status"].eq("PASS").all():
    raise ValueError("Sales target integrity validation has not passed.")

cl082_certified = df.loc[
    df["certified_sales_eligible"].eq(True)
].copy()

cl082_certified["target_month"] = (
    pd.to_datetime(cl082_certified["order_date"])
    .dt.strftime("%Y-%m")
)

cl082_actuals = (
    cl082_certified
    .groupby(
        ["target_month", "channel_id"],
        dropna=False,
        as_index=False
    )
    .agg(
        net_revenue_actual=("net_revenue", "sum"),
        gross_profit_actual=("gross_profit", "sum")
    )
)

cl082_targets = cl081_targets[
    [
        "target_month",
        "channel_id",
        "net_revenue_target",
        "gross_profit_target"
    ]
].copy()

cl082_comparison = cl082_actuals.merge(
    cl082_targets,
    on=["target_month", "channel_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

cl082_missing_channel = cl082_comparison[
    "channel_id"
].isna()

cl082_results = pd.DataFrame([
    {
        "control": "Actual records preserved after aggregation",
        "expected": len(cl082_actuals),
        "actual": len(cl082_comparison)
    },
    {
        "control": "Unmatched known month-channel combinations",
        "expected": 0,
        "actual": int(
            (
                cl082_comparison["_merge"].ne("both")
                & ~cl082_missing_channel
            ).sum()
        )
    },
    {
        "control": "Unknown channel records retained",
        "expected": int(
            cl082_actuals["channel_id"].isna().sum()
        ),
        "actual": int(cl082_missing_channel.sum())
    },
    {
        "control": "Net revenue preserved",
        "expected": round(
            cl082_certified["net_revenue"].sum(), 2
        ),
        "actual": round(
            cl082_comparison["net_revenue_actual"].sum(), 2
        )
    },
    {
        "control": "Gross profit preserved",
        "expected": round(
            cl082_certified["gross_profit"].sum(), 2
        ),
        "actual": round(
            cl082_comparison["gross_profit_actual"].sum(), 2
        )
    }
])

cl082_results["difference"] = (
    cl082_results["actual"]
    - cl082_results["expected"]
).round(2)

cl082_results["status"] = np.where(
    cl082_results["difference"].abs() <= 0.005,
    "PASS",
    "FAIL"
)

display(cl082_results)

print("\nJoin coverage:")
display(cl082_comparison["_merge"].value_counts())

print(
    "\nSales actuals vs targets reconciliation:",
    "PASS"
    if cl082_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,expected,actual,difference,status
0,Actual records preserved after aggregation,60.00,60.00,0.00,PASS
1,Unmatched known month-channel combinations,0.00,0.00,0.00,PASS
2,Unknown channel records retained,0.00,0.00,0.00,PASS
3,Net revenue preserved,"13,564,280.85","13,564,280.85",0.00,PASS
4,Gross profit preserved,"4,080,642.31","4,080,642.31",0.00,PASS



Join coverage:


_merge
both          60
left_only      0
right_only     0
Name: count, dtype: int64


Sales actuals vs targets reconciliation: PASS


In [85]:

# 08 — Validation Summary & Execution Gate
# Sales target coverage and reporting restrictions validation

cl083_matched = cl082_comparison.loc[
    cl082_comparison["_merge"].eq("both")
].copy()

cl083_expected_pairs = 60

cl083_checks = {
    "Expected matched month-channel pairs": (
        len(cl083_matched), cl083_expected_pairs
    ),
    "Missing net revenue targets": (
        int(cl083_matched["net_revenue_target"].isna().sum()), 0
    ),
    "Missing gross profit targets": (
        int(cl083_matched["gross_profit_target"].isna().sum()), 0
    ),
    "Duplicate reporting grain": (
        int(
            cl083_matched.duplicated(
                ["target_month", "channel_id"]
            ).sum()
        ), 0
    ),
    "Nonpositive net revenue targets": (
        int(
            (cl083_matched["net_revenue_target"] <= 0).sum()
        ), 0
    ),
    "Nonpositive gross profit targets": (
        int(
            (cl083_matched["gross_profit_target"] <= 0).sum()
        ), 0
    ),
    "Unexpected reporting months": (
        int(
            (~cl083_matched["target_month"].isin(
                cl081_expected_months
            )).sum()
        ), 0
    ),
    "Unexpected reporting channels": (
        int(
            (~cl083_matched["channel_id"].isin(
                cl081_channels
            )).sum()
        ), 0
    )
}

cl083_results = pd.DataFrame([
    {
        "control": control,
        "actual": actual,
        "expected": expected,
        "status": "PASS" if actual == expected else "FAIL"
    }
    for control, (actual, expected) in cl083_checks.items()
])

display(cl083_results)

print(
    "\nSales target reporting coverage:",
    "PASS"
    if cl083_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,actual,expected,status
0,Expected matched month-channel pairs,60,60,PASS
1,Missing net revenue targets,0,0,PASS
2,Missing gross profit targets,0,0,PASS
3,Duplicate reporting grain,0,0,PASS
4,Nonpositive net revenue targets,0,0,PASS
5,Nonpositive gross profit targets,0,0,PASS
6,Unexpected reporting months,0,0,PASS
7,Unexpected reporting channels,0,0,PASS



Sales target reporting coverage: PASS


In [86]:

# 08 — Validation Summary & Execution Gate
# Sales target validation summary

cl084_control_groups = {
    "Sales target grain and integrity": cl081_results,
    "Actuals vs targets reconciliation": cl082_results,
    "Sales target reporting coverage": cl083_results
}

cl084_expected_checks = {
    "Sales target grain and integrity": 12,
    "Actuals vs targets reconciliation": 5,
    "Sales target reporting coverage": 8
}

cl084_summary = pd.DataFrame([
    {
        "control": name,
        "expected_checks": cl084_expected_checks[name],
        "actual_checks": len(results),
        "passed": int(results["status"].eq("PASS").sum()),
        "failed": int(results["status"].eq("FAIL").sum()),
        "status": (
            "PASS"
            if len(results) == cl084_expected_checks[name]
            and results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for name, results in cl084_control_groups.items()
])

display(cl084_summary)

cl084_total_checks = int(cl084_summary["actual_checks"].sum())
cl084_total_passed = int(cl084_summary["passed"].sum())
cl084_total_failed = int(cl084_summary["failed"].sum())

cl084_status = (
    "PASS"
    if cl084_total_checks == 25
    and cl084_total_passed == 25
    and cl084_total_failed == 0
    and cl084_summary["status"].eq("PASS").all()
    else "FAIL"
)

print(f"\nTotal checks: {cl084_total_checks}")
print(f"Total passed: {cl084_total_passed}")
print(f"Total failed: {cl084_total_failed}")

print(f"\nSales target validation: {cl084_status}")


,control,expected_checks,actual_checks,passed,failed,status
0,Sales target grain and integrity,12,12,12,0,PASS
1,Actuals vs targets reconciliation,5,5,5,0,PASS
2,Sales target reporting coverage,8,8,8,0,PASS



Total checks: 25
Total passed: 25
Total failed: 0

Sales target validation: PASS


In [87]:

# 08 — Validation Summary & Execution Gate
# Business question dimensional dependency inspection

cl085_dimensions = [
    "dim_customer",
    "dim_product",
    "dim_channel",
    "dim_region",
    "dim_date"
]

cl085_results = []

for table in cl085_dimensions:

    schema = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        warehouse_connection
    )

    columns = schema["name"].tolist()

    cl085_results.append({
        "dimension": table,
        "column_count": len(columns),
        "available_columns": ", ".join(columns),
        "status": "PASS" if len(columns) > 0 else "FAIL"
    })

cl085_results = pd.DataFrame(cl085_results)

display(cl085_results)

print(
    "\nBusiness question dimension availability:",
    "PASS"
    if cl085_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension,column_count,available_columns,status
0,dim_customer,8,"customer_id, customer_name, customer_segment, ...",PASS
1,dim_product,9,"product_id, product_name, category, subcategor...",PASS
2,dim_channel,4,"channel_id, channel_name, channel_group, is_di...",PASS
3,dim_region,5,"region_id, country, region_name, market_tier, ...",PASS
4,dim_date,12,"date_key, full_date, year, quarter, month_numb...",PASS



Business question dimension availability: PASS


In [88]:

# 08 — Validation Summary & Execution Gate
# Business question attribute coverage validation

cl085b_requirements = {
    "BQ01 — Profitability Growth": {
        "dim_date": ["full_date", "year", "month_number"]
    },
    "BQ02 — Sales Channel Performance": {
        "dim_channel": ["channel_id", "channel_name"]
    },
    "BQ03 — Product & Category Performance": {
        "dim_product": ["product_id", "product_name", "category"]
    },
    "BQ04 — Customer Segment Performance": {
        "dim_customer": ["customer_id", "customer_segment"]
    },
    "BQ05 — Regional Performance": {
        "dim_region": ["region_id", "country", "region_name"]
    },
    "BQ06 — Discount Impact": {
        "dim_date": ["full_date", "year", "month_number"],
        "dim_channel": ["channel_id", "channel_name"],
        "dim_product": ["product_id", "category"]
    },
    "BQ07 — Management Recommendations": {
        "dim_customer": ["customer_segment"],
        "dim_product": ["category"],
        "dim_channel": ["channel_name"],
        "dim_region": ["country", "region_name"]
    }
}

cl085b_schema = {}

for table in [
    "dim_customer",
    "dim_product",
    "dim_channel",
    "dim_region",
    "dim_date"
]:
    schema = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        warehouse_connection
    )
    cl085b_schema[table] = set(schema["name"])

cl085b_results = []

for business_question, dimensions in cl085b_requirements.items():

    missing_attributes = []

    for table, required_columns in dimensions.items():
        for column in required_columns:
            if column not in cl085b_schema[table]:
                missing_attributes.append(f"{table}.{column}")

    cl085b_results.append({
        "business_question": business_question,
        "required_attributes": sum(
            len(columns) for columns in dimensions.values()
        ),
        "missing_attributes": len(missing_attributes),
        "missing_details": (
            ", ".join(missing_attributes)
            if missing_attributes else "None"
        ),
        "status": (
            "PASS" if not missing_attributes else "FAIL"
        )
    })

cl085b_results = pd.DataFrame(cl085b_results)

display(cl085b_results)

print(
    "\nBusiness question attribute coverage:",
    "PASS"
    if cl085b_results["status"].eq("PASS").all()
    else "FAIL"
)


,business_question,required_attributes,missing_attributes,missing_details,status
0,BQ01 — Profitability Growth,3,0,None,PASS
1,BQ02 — Sales Channel Performance,2,0,None,PASS
2,BQ03 — Product & Category Performance,3,0,None,PASS
3,BQ04 — Customer Segment Performance,2,0,None,PASS
4,BQ05 — Regional Performance,3,0,None,PASS
5,BQ06 — Discount Impact,7,0,None,PASS
6,BQ07 — Management Recommendations,5,0,None,PASS



Business question attribute coverage: PASS


In [89]:

# 08 — Validation Summary & Execution Gate
# Business dimension attribute quality validation

cl085c_attribute_requirements = {
    "dim_customer": ["customer_segment"],
    "dim_product": ["product_name", "category"],
    "dim_channel": ["channel_name"],
    "dim_region": ["country", "region_name"],
    "dim_date": ["full_date", "year", "month_number"]
}

cl085c_results = []

for table, attributes in cl085c_attribute_requirements.items():

    dimension_df = pd.read_sql_query(
        f'SELECT * FROM "{table}"',
        warehouse_connection
    )

    for attribute in attributes:

        values = dimension_df[attribute]

        null_count = int(values.isna().sum())

        blank_count = int(
            values.dropna()
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

        cl085c_results.append({
            "dimension": table,
            "attribute": attribute,
            "dimension_records": len(dimension_df),
            "null_values": null_count,
            "blank_values": blank_count,
            "status": (
                "PASS"
                if null_count == 0 and blank_count == 0
                else "FAIL"
            )
        })

cl085c_results = pd.DataFrame(cl085c_results)

display(cl085c_results)

print(
    "\nBusiness dimension attribute quality:",
    "PASS"
    if cl085c_results["status"].eq("PASS").all()
    else "FAIL"
)


,dimension,attribute,dimension_records,null_values,blank_values,status
0,dim_customer,customer_segment,2500,0,0,PASS
1,dim_product,product_name,180,0,0,PASS
2,dim_product,category,180,0,0,PASS
3,dim_channel,channel_name,5,0,0,PASS
4,dim_region,country,8,0,0,PASS
5,dim_region,region_name,8,0,0,PASS
6,dim_date,full_date,546,0,0,PASS
7,dim_date,year,546,0,0,PASS
8,dim_date,month_number,546,0,0,PASS



Business dimension attribute quality: PASS


In [90]:

# 08 — Validation Summary & Execution Gate
# Descriptive dimensional join integrity validation

cl085d_join_config = {
    "dim_customer": ("customer_id", "customer_id", ["customer_segment"]),
    "dim_product": ("product_id", "product_id", ["product_name", "category"]),
    "dim_channel": ("channel_id", "channel_id", ["channel_name"]),
    "dim_region": ("region_id", "region_id", ["country", "region_name"]),
    "dim_date": ("order_date", "full_date", ["year", "month_number"])
}

cl085d_joined = df.copy()
cl085d_join_results = []

for table, (left_key, right_key, attributes) in cl085d_join_config.items():

    dimension_df = pd.read_sql_query(
        f'SELECT * FROM "{table}"',
        warehouse_connection
    )

    dimension_df = dimension_df[
        [right_key] + attributes
    ].copy()

    dimension_df = dimension_df.rename(
        columns={
            attribute: f"{table}_{attribute}"
            for attribute in attributes
        }
    )

    rows_before = len(cl085d_joined)

    cl085d_joined = cl085d_joined.merge(
        dimension_df,
        how="left",
        left_on=left_key,
        right_on=right_key,
        validate="many_to_one",
        indicator=f"_join_{table}"
    )

    unmatched_non_null = int(
        (
            cl085d_joined[left_key].notna()
            & cl085d_joined[f"_join_{table}"].eq("left_only")
        ).sum()
    )

    cl085d_join_results.append({
        "dimension": table,
        "rows_before": rows_before,
        "rows_after": len(cl085d_joined),
        "row_difference": len(cl085d_joined) - rows_before,
        "unmatched_non_null": unmatched_non_null,
        "status": (
            "PASS"
            if len(cl085d_joined) == rows_before
            and unmatched_non_null == 0
            else "FAIL"
        )
    })

cl085d_join_results = pd.DataFrame(cl085d_join_results)

cl085d_financial_columns = [
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "total_cost",
    "gross_profit"
]

cl085d_original_certified = df.loc[
    df["certified_sales_eligible"].eq(True)
]

cl085d_joined_certified = cl085d_joined.loc[
    cl085d_joined["certified_sales_eligible"].eq(True)
]

cl085d_financial_results = pd.DataFrame([
    {
        "metric": metric,
        "original": round(
            cl085d_original_certified[metric].sum(), 2
        ),
        "joined": round(
            cl085d_joined_certified[metric].sum(), 2
        )
    }
    for metric in cl085d_financial_columns
])

cl085d_financial_results["difference"] = (
    cl085d_financial_results["joined"]
    - cl085d_financial_results["original"]
).round(2)

cl085d_financial_results["status"] = np.where(
    cl085d_financial_results["difference"].abs() <= 0.005,
    "PASS",
    "FAIL"
)

cl085d_record_checks = {
    "Analytical row count preserved": (
        len(cl085d_joined) == len(df)
    ),
    "Analytical row keys remain unique": (
        cl085d_joined["analytical_row_key"].is_unique
    ),
    "Certified population preserved": (
        len(cl085d_joined_certified)
        == len(cl085d_original_certified)
    ),
    "Missing customer reference preserved": (
        cl085d_joined["customer_id"].isna().sum()
        == df["customer_id"].isna().sum()
    ),
    "Missing channel reference preserved": (
        cl085d_joined["channel_id"].isna().sum()
        == df["channel_id"].isna().sum()
    )
}

cl085d_record_results = pd.DataFrame([
    {
        "control": control,
        "status": "PASS" if passed else "FAIL"
    }
    for control, passed in cl085d_record_checks.items()
])

display(cl085d_join_results)
display(cl085d_financial_results)
display(cl085d_record_results)

cl085d_status = (
    "PASS"
    if (
        cl085d_join_results["status"].eq("PASS").all()
        and cl085d_financial_results["status"].eq("PASS").all()
        and cl085d_record_results["status"].eq("PASS").all()
    )
    else "FAIL"
)

print(f"\nDescriptive dimensional join integrity: {cl085d_status}")


,dimension,rows_before,rows_after,row_difference,unmatched_non_null,status
0,dim_customer,23530,23530,0,0,PASS
1,dim_product,23530,23530,0,0,PASS
2,dim_channel,23530,23530,0,0,PASS
3,dim_region,23530,23530,0,0,PASS
4,dim_date,23530,23530,0,0,PASS


,metric,original,joined,difference,status
0,gross_revenue,"16,068,562.07","16,068,562.07",0.00,PASS
1,discount_amount,"2,504,281.22","2,504,281.22",0.00,PASS
2,net_revenue,"13,564,280.85","13,564,280.85",0.00,PASS
3,total_cost,"9,483,638.54","9,483,638.54",0.00,PASS
4,gross_profit,"4,080,642.31","4,080,642.31",0.00,PASS


,control,status
0,Analytical row count preserved,PASS
1,Analytical row keys remain unique,PASS
2,Certified population preserved,PASS
3,Missing customer reference preserved,PASS
4,Missing channel reference preserved,PASS



Descriptive dimensional join integrity: PASS


In [91]:

# 08 — Validation Summary & Execution Gate
# Descriptive attribution and reporting coverage validation

cl085e_certified = cl085d_joined.loc[
    cl085d_joined["certified_sales_eligible"].eq(True)
].copy()

cl085e_required_attributes = [
    "dim_customer_customer_segment",
    "dim_product_product_name",
    "dim_product_category",
    "dim_channel_channel_name",
    "dim_region_country",
    "dim_region_region_name",
    "dim_date_year",
    "dim_date_month_number"
]

cl085e_results = []

for attribute in cl085e_required_attributes:

    missing_values = int(
        cl085e_certified[attribute].isna().sum()
    )

    cl085e_results.append({
        "control": f"Certified coverage: {attribute}",
        "expected": 0,
        "actual": missing_values,
        "status": "PASS" if missing_values == 0 else "REVIEW"
    })

cl085e_exception_checks = {
    "Unknown Customer attribution preserved": (
        int(
            cl085d_joined["customer_attribution"]
            .eq("Unknown Customer").sum()
        ),
        int(df["customer_id"].isna().sum())
    ),
    "Unknown Channel attribution preserved": (
        int(
            cl085d_joined["channel_attribution"]
            .eq("Unknown Channel").sum()
        ),
        int(df["channel_id"].isna().sum())
    )
}

for control, (actual, expected) in cl085e_exception_checks.items():

    cl085e_results.append({
        "control": control,
        "expected": expected,
        "actual": actual,
        "status": "PASS" if actual == expected else "FAIL"
    })

cl085e_results = pd.DataFrame(cl085e_results)

display(cl085e_results)

print(
    "\nDescriptive attribution coverage:",
    "PASS"
    if cl085e_results["status"].eq("PASS").all()
    else "REVIEW REQUIRED"
)


,control,expected,actual,status
0,Certified coverage: dim_customer_customer_segment,0,1,REVIEW
1,Certified coverage: dim_product_product_name,0,0,PASS
2,Certified coverage: dim_product_category,0,0,PASS
3,Certified coverage: dim_channel_channel_name,0,0,PASS
4,Certified coverage: dim_region_country,0,0,PASS
5,Certified coverage: dim_region_region_name,0,0,PASS
6,Certified coverage: dim_date_year,0,0,PASS
7,Certified coverage: dim_date_month_number,0,0,PASS
8,Unknown Customer attribution preserved,1,1,PASS
9,Unknown Channel attribution preserved,1,1,PASS



Descriptive attribution coverage: REVIEW REQUIRED


In [92]:

# 08 — Validation Summary & Execution Gate
# Approved customer attribution exception validation (CL006)

cl085f_certified = cl085d_joined.loc[
    cl085d_joined["certified_sales_eligible"].eq(True)
].copy()

cl085f_missing_segment = (
    cl085f_certified["dim_customer_customer_segment"].isna()
)

cl085f_approved_unknown = (
    cl085f_certified["customer_id"].isna()
    & cl085f_certified["customer_attribution"].eq("Unknown Customer")
)

cl085f_results = pd.DataFrame([
    {
        "control": "Missing certified customer segments",
        "expected": 1,
        "actual": int(cl085f_missing_segment.sum())
    },
    {
        "control": "Approved Unknown Customer records",
        "expected": 1,
        "actual": int(cl085f_approved_unknown.sum())
    },
    {
        "control": "Unapproved missing customer segments",
        "expected": 0,
        "actual": int(
            (cl085f_missing_segment & ~cl085f_approved_unknown).sum()
        )
    },
    {
        "control": "Approved exceptions with populated segments",
        "expected": 0,
        "actual": int(
            (cl085f_approved_unknown & ~cl085f_missing_segment).sum()
        )
    }
])

cl085f_results["status"] = np.where(
    cl085f_results["expected"].eq(cl085f_results["actual"]),
    "PASS",
    "FAIL"
)

cl085f_certified["customer_segment_reporting"] = (
    cl085f_certified["dim_customer_customer_segment"]
    .fillna("Unknown Customer")
)

cl085f_reporting_checks = pd.DataFrame([
    {
        "control": "Reporting segments fully attributed",
        "status": (
            "PASS"
            if cl085f_certified["customer_segment_reporting"].notna().all()
            else "FAIL"
        )
    },
    {
        "control": "Certified record count preserved",
        "status": (
            "PASS"
            if len(cl085f_certified) == 21342
            else "FAIL"
        )
    },
    {
        "control": "Certified net revenue preserved",
        "status": (
            "PASS"
            if abs(
                cl085f_certified["net_revenue"].sum()
                - 13564280.85
            ) <= 0.01
            else "FAIL"
        )
    }
])

display(cl085f_results)
display(cl085f_reporting_checks)

cl085f_status = (
    "PASS"
    if cl085f_results["status"].eq("PASS").all()
    and cl085f_reporting_checks["status"].eq("PASS").all()
    else "FAIL"
)

print(f"\nApproved customer attribution exception: {cl085f_status}")


,control,expected,actual,status
0,Missing certified customer segments,1,1,PASS
1,Approved Unknown Customer records,1,1,PASS
2,Unapproved missing customer segments,0,0,PASS
3,Approved exceptions with populated segments,0,0,PASS


,control,status
0,Reporting segments fully attributed,PASS
1,Certified record count preserved,PASS
2,Certified net revenue preserved,PASS



Approved customer attribution exception: PASS


In [93]:

# 08 — Validation Summary & Execution Gate
# Business question readiness validation summary

cl086_control_groups = {
    "Dimension availability": cl085_results,
    "Business question attribute coverage": cl085b_results,
    "Dimension attribute quality": cl085c_results,
    "Descriptive join integrity": cl085d_join_results,
    "Financial preservation": cl085d_financial_results,
    "Record preservation": cl085d_record_results,
    "Approved CL006 exception": cl085f_results,
    "CL006 reporting integrity": cl085f_reporting_checks
}

cl086_expected_checks = {
    "Dimension availability": 5,
    "Business question attribute coverage": 7,
    "Dimension attribute quality": 9,
    "Descriptive join integrity": 5,
    "Financial preservation": 5,
    "Record preservation": 5,
    "Approved CL006 exception": 4,
    "CL006 reporting integrity": 3
}

cl086_summary = pd.DataFrame([
    {
        "control": name,
        "expected_checks": cl086_expected_checks[name],
        "actual_checks": len(results),
        "passed": int(results["status"].eq("PASS").sum()),
        "failed": int(results["status"].eq("FAIL").sum()),
        "status": (
            "PASS"
            if len(results) == cl086_expected_checks[name]
            and results["status"].eq("PASS").all()
            else "FAIL"
        )
    }
    for name, results in cl086_control_groups.items()
])

cl086_total_checks = int(cl086_summary["actual_checks"].sum())
cl086_total_passed = int(cl086_summary["passed"].sum())
cl086_total_failed = int(cl086_summary["failed"].sum())

cl086_status = (
    "PASS"
    if cl086_total_checks == 43
    and cl086_total_passed == 43
    and cl086_total_failed == 0
    and cl086_summary["status"].eq("PASS").all()
    else "FAIL"
)

display(cl086_summary)

print(f"\nTotal checks: {cl086_total_checks}")
print(f"Total passed: {cl086_total_passed}")
print(f"Total failed: {cl086_total_failed}")
print(f"\nBusiness question technical readiness: {cl086_status}")


,control,expected_checks,actual_checks,passed,failed,status
0,Dimension availability,5,5,5,0,PASS
1,Business question attribute coverage,7,7,7,0,PASS
2,Dimension attribute quality,9,9,9,0,PASS
3,Descriptive join integrity,5,5,5,0,PASS
4,Financial preservation,5,5,5,0,PASS
5,Record preservation,5,5,5,0,PASS
6,Approved CL006 exception,4,4,4,0,PASS
7,CL006 reporting integrity,3,3,3,0,PASS



Total checks: 43
Total passed: 43
Total failed: 0

Business question technical readiness: PASS


In [94]:

# 08 — Validation Summary & Execution Gate
# Outstanding restrictions and dependencies register

cl087_restrictions = [
    {
        "clarification": "CL001",
        "issue": "Unreliable order identity",
        "analytical_treatment": "Order-based KPIs blocked",
        "source_status": "OPEN",
        "reporting_impact": "RESTRICTED"
    },
    {
        "clarification": "CL002",
        "issue": "Unconfirmed partial-return amounts",
        "analytical_treatment": "Provisional financial population",
        "source_status": "OPEN",
        "reporting_impact": "PROVISIONAL"
    },
    {
        "clarification": "CL003",
        "issue": "Unmapped payment method",
        "analytical_treatment": "Unmapped reporting category",
        "source_status": "OPEN",
        "reporting_impact": "CONTROLLED"
    },
    {
        "clarification": "CL004",
        "issue": "Duplicate source sales-line identifiers",
        "analytical_treatment": "Unique analytical row key",
        "source_status": "OPEN",
        "reporting_impact": "CONTROLLED"
    },
    {
        "clarification": "CL005",
        "issue": "Invalid numeric source values",
        "analytical_treatment": "Exclude invalid records from financial KPIs",
        "source_status": "OPEN",
        "reporting_impact": "CONTROLLED"
    },
    {
        "clarification": "CL006",
        "issue": "Missing customer/channel references",
        "analytical_treatment": "Approved Unknown attribution",
        "source_status": "OPEN",
        "reporting_impact": "CONTROLLED"
    },
    {
        "clarification": "CL007",
        "issue": "High-cost outliers and negative gross profit",
        "analytical_treatment": "Retain, flag, and investigate",
        "source_status": "OPEN",
        "reporting_impact": "INVESTIGATION"
    }
]

cl087_results = pd.DataFrame(cl087_restrictions)

cl087_expected_ids = {
    f"CL{i:03d}" for i in range(1, 8)
}

cl087_checks = {
    "All seven clarifications documented": (
        set(cl087_results["clarification"]) == cl087_expected_ids
    ),
    "No duplicate clarification IDs": (
        cl087_results["clarification"].is_unique
    ),
    "All source issues remain explicitly tracked": (
        cl087_results["source_status"].eq("OPEN").all()
    ),
    "No missing analytical treatments": (
        cl087_results["analytical_treatment"].notna().all()
        and cl087_results["analytical_treatment"].str.strip().ne("").all()
    ),
    "No missing reporting impacts": (
        cl087_results["reporting_impact"].notna().all()
        and cl087_results["reporting_impact"].str.strip().ne("").all()
    )
}

cl087_validation = pd.DataFrame([
    {
        "control": control,
        "status": "PASS" if passed else "FAIL"
    }
    for control, passed in cl087_checks.items()
])

display(cl087_results)
display(cl087_validation)

print(
    "\nOutstanding restrictions register:",
    "PASS"
    if cl087_validation["status"].eq("PASS").all()
    else "FAIL"
)


,clarification,issue,analytical_treatment,source_status,reporting_impact
0,CL001,Unreliable order identity,Order-based KPIs blocked,OPEN,RESTRICTED
1,CL002,Unconfirmed partial-return amounts,Provisional financial population,OPEN,PROVISIONAL
2,CL003,Unmapped payment method,Unmapped reporting category,OPEN,CONTROLLED
3,CL004,Duplicate source sales-line identifiers,Unique analytical row key,OPEN,CONTROLLED
4,CL005,Invalid numeric source values,Exclude invalid records from financial KPIs,OPEN,CONTROLLED
5,CL006,Missing customer/channel references,Approved Unknown attribution,OPEN,CONTROLLED
6,CL007,High-cost outliers and negative gross profit,"Retain, flag, and investigate",OPEN,INVESTIGATION


,control,status
0,All seven clarifications documented,PASS
1,No duplicate clarification IDs,PASS
2,All source issues remain explicitly tracked,PASS
3,No missing analytical treatments,PASS
4,No missing reporting impacts,PASS



Outstanding restrictions register: PASS


In [95]:

# 08 — Validation Summary & Execution Gate
# Final analytical execution gate assessment

cl088_evidence = {
    "Dimensional and KPI validation": (
        cl078_results["status"].eq("PASS").all()
    ),
    "Derived KPI validation": (
        cl079b_results["status"].eq("PASS").all()
    ),
    "Units Sold dimensional reconciliation": (
        cl079c_results["status"].eq("PASS").all()
    ),
    "Extended KPI permissions": (
        cl079d_validation["status"].eq("PASS").all()
    ),
    "Sales target validation": (
        cl084_status == "PASS"
    ),
    "Business question technical readiness": (
        cl086_status == "PASS"
    ),
    "Outstanding restrictions documented": (
        cl087_validation["status"].eq("PASS").all()
    )
}

cl088_evidence_results = pd.DataFrame([
    {
        "requirement": requirement,
        "status": "PASS" if passed else "FAIL"
    }
    for requirement, passed in cl088_evidence.items()
])

cl088_critical_controls_pass = all(
    cl088_evidence.values()
)

cl088_open_restrictions = (
    cl087_results["source_status"].eq("OPEN").any()
)

if not cl088_critical_controls_pass:
    cl088_decision = "NO-GO"
elif cl088_open_restrictions:
    cl088_decision = "CONDITIONAL GO"
else:
    cl088_decision = "GO"

cl088_authorized_scope = {
    "Certified financial analysis": "AUTHORIZED",
    "Certified dimensional analysis": "AUTHORIZED",
    "Certified Units Sold analysis": "AUTHORIZED",
    "Sales actuals vs targets": "AUTHORIZED — APPROVED GRAIN ONLY",
    "Provisional return-inclusive analysis": "PROVISIONAL ONLY",
    "Order Count / AOV / Order Frequency": "BLOCKED",
    "Unknown Customer / Channel attribution": "REQUIRED",
    "Outlier and negative-profit investigation": "AUTHORIZED — NO AUTOMATIC EXCLUSION"
}

cl088_scope_results = pd.DataFrame([
    {
        "analytical_area": area,
        "execution_permission": permission
    }
    for area, permission in cl088_authorized_scope.items()
])

display(cl088_evidence_results)
display(cl088_scope_results)

print(f"\nExecution Gate Decision: {cl088_decision}")
print(
    "Critical validation controls:",
    "PASS" if cl088_critical_controls_pass else "FAIL"
)
print(
    "Outstanding source restrictions:",
    "YES" if cl088_open_restrictions else "NO"
)


,requirement,status
0,Dimensional and KPI validation,PASS
1,Derived KPI validation,PASS
2,Units Sold dimensional reconciliation,PASS
3,Extended KPI permissions,PASS
4,Sales target validation,PASS
5,Business question technical readiness,PASS
6,Outstanding restrictions documented,PASS


,analytical_area,execution_permission
0,Certified financial analysis,AUTHORIZED
1,Certified dimensional analysis,AUTHORIZED
2,Certified Units Sold analysis,AUTHORIZED
3,Sales actuals vs targets,AUTHORIZED — APPROVED GRAIN ONLY
4,Provisional return-inclusive analysis,PROVISIONAL ONLY
5,Order Count / AOV / Order Frequency,BLOCKED
6,Unknown Customer / Channel attribution,REQUIRED
7,Outlier and negative-profit investigation,AUTHORIZED — NO AUTOMATIC EXCLUSION



Execution Gate Decision: CONDITIONAL GO
Critical validation controls: PASS
Outstanding source restrictions: YES


In [96]:

# 08 — Validation Summary & Execution Gate
# Earlier validation evidence inventory

cl089_candidates = []

for name, value in globals().copy().items():

    if name.startswith("_"):
        continue

    if not isinstance(value, pd.DataFrame):
        continue

    name_lower = name.lower()

    if not any(
        keyword in name_lower
        for keyword in [
            "result",
            "summary",
            "validation",
            "audit",
            "check"
        ]
    ):
        continue

    cl089_candidates.append({
        "variable": name,
        "rows": len(value),
        "columns": ", ".join(value.columns.astype(str)),
        "has_status": "status" in value.columns
    })

cl089_inventory = pd.DataFrame(cl089_candidates)

if not cl089_inventory.empty:
    cl089_inventory = cl089_inventory.sort_values(
        "variable"
    ).reset_index(drop=True)

display(cl089_inventory)

print(
    "\nValidation evidence inventory completed:",
    "PASS"
)


,variable,rows,columns,has_status
0,cl001_candidate_results,2,"check, expected, observed, status",True
1,cl001_checks,3,"check, observed, expected, status",True
2,cl002_results,4,"metric, expected, observed, difference, status",True
3,cl002_summary,6,"control, checks, passed, failed, status",True
4,cl003_summary,4,"control, status",True
...,...,...,...,...
67,dimensional_validation_summary,7,"control, checks, passed, failed, status",True
68,population_results,2,"population, expected_count, actual_count, mism...",False
69,results,6,"control, checks, passed, failed, status",True
70,schema_result,23,"column, present, dtype",False



Validation evidence inventory completed: PASS


In [97]:

# 08 — Validation Summary & Execution Gate
# Earlier validation evidence classification

cl090_excluded_prefixes = (
    "cl07",
    "cl08",
    "dimension_"
)

cl090_records = []

for variable_name in cl089_inventory["variable"]:

    if variable_name.startswith(cl090_excluded_prefixes):
        continue

    value = globals().get(variable_name)

    if not isinstance(value, pd.DataFrame):
        continue

    if "status" not in value.columns:
        continue

    statuses = (
        value["status"]
        .astype("string")
        .str.upper()
        .str.strip()
    )

    cl090_records.append({
        "variable": variable_name,
        "checks": len(value),
        "passed": int(statuses.eq("PASS").sum()),
        "failed": int(statuses.eq("FAIL").sum()),
        "other_statuses": int(
            (~statuses.isin(["PASS", "FAIL"])).sum()
        ),
        "observed_statuses": ", ".join(
            sorted(statuses.dropna().unique())
        )
    })

cl090_results = pd.DataFrame(cl090_records)

if not cl090_results.empty:
    cl090_results = cl090_results.sort_values(
        "variable"
    ).reset_index(drop=True)

pd.set_option("display.max_rows", 100)

display(cl090_results)

print(
    "\nCandidate evidence tables:",
    len(cl090_results)
)

print(
    "Evidence classification:",
    "COMPLETED"
)


,variable,checks,passed,failed,other_statuses,observed_statuses
0,cl001_candidate_results,2,2,0,0,PASS
1,cl001_checks,3,3,0,0,PASS
2,cl002_results,4,4,0,0,PASS
3,cl002_summary,6,6,0,0,PASS
4,cl003_summary,4,4,0,0,PASS
5,cl004_collision_results,3,3,0,0,PASS
6,cl004_duplicate_results,3,3,0,0,PASS
7,cl004_eligibility_results,2,2,0,0,PASS
8,cl004_flag_results,2,2,0,0,PASS
9,cl004_key_results,4,4,0,0,PASS



Candidate evidence tables: 32
Evidence classification: COMPLETED


In [98]:

# 08 — Validation Summary & Execution Gate
# Earlier sections evidence traceability inspection

cl091_evidence_map = {
    "CL001": ["cl001_checks", "cl001_candidate_results"],
    "CL002": ["cl002_summary"],
    "CL003": ["cl003_summary"],
    "CL004": ["cl004_summary"],
    "CL005": ["cl005_summary"],
    "CL006": ["cl006_summary"],
    "CL007": ["cl007_summary"]
}

cl091_records = []

for clarification, variables in cl091_evidence_map.items():

    for variable_name in variables:

        evidence = globals().get(variable_name)

        available = isinstance(evidence, pd.DataFrame)

        if available and "status" in evidence.columns:

            statuses = (
                evidence["status"]
                .astype("string")
                .str.upper()
                .str.strip()
            )

            checks = len(evidence)
            passed = int(statuses.eq("PASS").sum())
            failed = int(statuses.eq("FAIL").sum())

            status = (
                "PASS"
                if checks > 0 and passed == checks
                else "REVIEW"
            )

        else:
            checks = 0
            passed = 0
            failed = 0
            status = "MISSING"

        cl091_records.append({
            "clarification": clarification,
            "evidence_variable": variable_name,
            "checks": checks,
            "passed": passed,
            "failed": failed,
            "status": status
        })

cl091_results = pd.DataFrame(cl091_records)

display(cl091_results)

print(
    "\nClarification evidence traceability:",
    "PASS"
    if cl091_results["status"].eq("PASS").all()
    else "REVIEW REQUIRED"
)


,clarification,evidence_variable,checks,passed,failed,status
0,CL001,cl001_checks,3,3,0,PASS
1,CL001,cl001_candidate_results,2,2,0,PASS
2,CL002,cl002_summary,6,6,0,PASS
3,CL003,cl003_summary,4,4,0,PASS
4,CL004,cl004_summary,6,6,0,PASS
5,CL005,cl005_summary,4,4,0,PASS
6,CL006,cl006_summary,5,5,0,PASS
7,CL007,cl007_summary,4,4,0,PASS



Clarification evidence traceability: PASS


In [99]:

# 08 — Validation Summary & Execution Gate
# Foundational validation evidence inspection

cl092_candidates = [
    "schema_result",
    "validation_df",
    "population_results",
    "results"
]

cl092_records = []

for variable_name in cl092_candidates:

    evidence = globals().get(variable_name)

    if not isinstance(evidence, pd.DataFrame):
        cl092_records.append({
            "variable": variable_name,
            "available": False,
            "rows": 0,
            "columns": "N/A",
            "status_values": "N/A"
        })
        continue

    status_columns = [
        column for column in evidence.columns
        if any(
            keyword in column.lower()
            for keyword in ["status", "valid", "match", "pass"]
        )
    ]

    status_details = []

    for column in status_columns:
        values = (
            evidence[column]
            .astype(str)
            .value_counts(dropna=False)
            .to_dict()
        )
        status_details.append(f"{column}: {values}")

    cl092_records.append({
        "variable": variable_name,
        "available": True,
        "rows": len(evidence),
        "columns": ", ".join(evidence.columns.astype(str)),
        "status_values": (
            " | ".join(status_details)
            if status_details else "No status field"
        )
    })

cl092_results = pd.DataFrame(cl092_records)

pd.set_option("display.max_colwidth", 300)

display(cl092_results)

print("\nFoundational evidence inspection: COMPLETED")


,variable,available,rows,columns,status_values
0,schema_result,True,23,"column, present, dtype",No status field
1,validation_df,True,13,"column, expected_type, actual_type, type_valid, null_count",type_valid: {'True': 13}
2,population_results,True,2,"population, expected_count, actual_count, mismatches",mismatches: {'0': 2}
3,results,True,6,"control, checks, passed, failed, status","passed: {'7': 2, '3': 1, '4': 1, '5': 1, '10': 1} | status: {'PASS': 6}"



Foundational evidence inspection: COMPLETED


In [100]:

# 08 — Validation Summary & Execution Gate
# Foundational evidence acceptance validation

cl093_checks = {
    "Required schema fields present": (
        len(schema_result) > 0
        and schema_result["present"].eq(True).all()
    ),
    "Required data types validated": (
        len(validation_df) > 0
        and validation_df["type_valid"].eq(True).all()
    ),
    "Population counts reconciled": (
        len(population_results) > 0
        and population_results["mismatches"].eq(0).all()
        and population_results["expected_count"]
            .eq(population_results["actual_count"]).all()
    ),
    "Existing validation groups passed": (
        len(results) > 0
        and results["status"].eq("PASS").all()
        and results["failed"].eq(0).all()
        and results["checks"].eq(results["passed"]).all()
    )
}

cl093_results = pd.DataFrame([
    {
        "control": control,
        "status": "PASS" if passed else "FAIL"
    }
    for control, passed in cl093_checks.items()
])

display(cl093_results)

print(
    "\nFoundational evidence acceptance:",
    "PASS"
    if cl093_results["status"].eq("PASS").all()
    else "FAIL"
)


,control,status
0,Required schema fields present,PASS
1,Required data types validated,PASS
2,Population counts reconciled,PASS
3,Existing validation groups passed,PASS



Foundational evidence acceptance: PASS


In [101]:

# 08 — Validation Summary & Execution Gate
# Final validation evidence and execution gate consolidation

cl094_evidence = {
    "Foundational validation evidence": (
        cl093_results["status"].eq("PASS").all()
    ),
    "Business clarification traceability": (
        cl091_results["status"].eq("PASS").all()
    ),
    "Dimensional and KPI controls": (
        cl088_evidence_results["status"].eq("PASS").all()
    ),
    "Sales target validation": (
        cl084_status == "PASS"
    ),
    "Business question technical readiness": (
        cl086_status == "PASS"
    ),
    "Outstanding restrictions register": (
        cl087_validation["status"].eq("PASS").all()
    )
}

cl094_results = pd.DataFrame([
    {
        "evidence_area": area,
        "status": "PASS" if passed else "FAIL"
    }
    for area, passed in cl094_evidence.items()
])

cl094_all_passed = all(cl094_evidence.values())

cl094_open_issues = (
    cl087_results["source_status"].eq("OPEN").any()
)

if not cl094_all_passed:
    cl094_decision = "NO-GO"
elif cl094_open_issues:
    cl094_decision = "CONDITIONAL GO"
else:
    cl094_decision = "GO"

cl094_consistency = (
    cl094_decision == cl088_decision
)

display(cl094_results)

print(f"\nEvidence areas: {len(cl094_results)}")
print(f"Evidence areas passed: {cl094_results['status'].eq('PASS').sum()}")
print(f"Evidence areas failed: {cl094_results['status'].eq('FAIL').sum()}")

print(f"\nFinal Execution Gate Decision: {cl094_decision}")
print(
    "Consistent with preliminary gate:",
    "PASS" if cl094_consistency else "FAIL"
)
print(
    "Outstanding source issues:",
    "YES" if cl094_open_issues else "NO"
)


,evidence_area,status
0,Foundational validation evidence,PASS
1,Business clarification traceability,PASS
2,Dimensional and KPI controls,PASS
3,Sales target validation,PASS
4,Business question technical readiness,PASS
5,Outstanding restrictions register,PASS



Evidence areas: 6
Evidence areas passed: 6
Evidence areas failed: 0

Final Execution Gate Decision: CONDITIONAL GO
Consistent with preliminary gate: PASS
Outstanding source issues: YES


## Validation Closure & Analytical Execution Checkpoint

**Project:** Nova Corporation — BR001 Executive Performance Review  
**Checkpoint:** Analytical Validation Complete  
**Execution Gate Decision:** CONDITIONAL GO  
**Governing References:** IRDS-BR001; BR001 Business Clarification Response & Approved Cleaning Decisions; BR001 Analytical Plan.

### Validation Outcome

The controlled analytical dataset has completed the required structural, population, financial, dimensional, KPI eligibility, and business-rule compliance validations within the approved BR001 scope.

The validation process confirmed:

- Preservation of the controlled analytical population of **23,530 records**.
- Reconciliation of **21,342 certified sales records** and the approved provisional population.
- Integrity of certified financial KPIs and authorized derived metrics.
- Dimensional join integrity, reporting attribution, and aggregation consistency.
- Sales target compatibility at the approved **month × channel, All Markets** grain.
- Technical data and attribute availability for business questions BQ01–BQ07.
- Traceability of approved treatments for clarifications CL001–CL007.
- Consistency of the final execution decision following notebook reproducibility verification.

### Outstanding Restrictions

The approved analytical treatments do not constitute resolution of the underlying source-data issues.

The following restrictions remain in effect:

- **CL001:** Order Count, Average Order Value, and Order Frequency remain blocked.
- **CL002:** Partial-return-inclusive financial results remain provisional.
- **CL003–CL006:** Approved source-preservation, eligibility, and reporting-attribution treatments must be maintained.
- **CL007:** High-cost outliers and negative-profit observations remain subject to business investigation without automatic exclusion.
- **Sales Targets:** Actual-versus-target comparisons are restricted to the approved reporting grain and compatible aggregations.

### Execution Gate Decision

**CONDITIONAL GO — Approved for Business Analysis**

The validated dataset is authorized for descriptive, diagnostic, and subsequent analytical work within the approved KPI definitions, populations, dimensional relationships, and reporting restrictions.

This decision does not authorize restricted KPIs, unsupported attribution, or unapproved changes to business rules.

### Checkpoint & Next Phase

This notebook serves as the validation evidence checkpoint for BR001. Subsequent business analysis must reference the approved analytical dataset and preserve the documented restrictions.

**Next Phase:** Business Analysis  
**Starting Point:** BQ01 — Profitability Growth

**Checkpoint Status:** VALIDATION COMPLETE — CONDITIONAL GO